# 12주차 실습. 조건을 넣고, 더 세게 밀기
*Week 12 lab. Feed the condition, then push harder*

**제출하지 않습니다. 채점하지 않습니다.** 과제 3(DDPM: Denoising Diffusion Probabilistic Model, 11/27 마감)과는 별개입니다.
*Not submitted, not graded. Separate from Assignment 3 (DDPM, due 27 November).*

9주차 노트북을 그대로 이어 갑니다. 같은 잡음 요리법, 같은 꼴의 신경망, 같은 역방향 한 스텝입니다. 오늘 새로 하는 일은 둘뿐입니다. **조건(클래스 번호)을 신경망에 넣고**, 표집할 때 **조건이 바꾼 방향으로 더 세게 밉니다**(classifier-free guidance, CFG).
*This continues the Week 9 notebook: the same noise recipe, the same network shape, the same reverse step. Only two things are new: we **feed a condition (the class index) to the network**, and at sampling time we **push harder along the direction the condition changes** (classifier-free guidance, CFG).*

## 먼저 큰 그림 / First, the four different jobs
9주차에는 **잡음에서 표본을 만드는 역방향 반복**을 배웠습니다. 이번에는 그 표본에 원하는 조건을 주고, 조건을 얼마나 강조할지 선택합니다. 계산을 더 작은 표현에서 하는 방법과 만들어진 표본의 쓰임을 평가하는 일은 또 다른 문제입니다.
*Week 9 taught the reverse process that generates samples from noise. This week adds a request and a choice about how strongly to emphasize it. A cheaper representation and a useful downstream result are separate questions.*

| 이름 / Name | 맡은 일 / Job | 이 실습 / Here |
|---|---|---|
| Conditioning, 조건 입력 | 클래스 y를 잡음 예측기에 알려 준다 / give the denoiser the requested class | §2–§3에서 구현 |
| CFG, **Classifier-Free Guidance** | 같은 망의 조건 있음·없음 예측을 조합한다 / combine two predictions from one network | §4의 한 줄 |
| DDPM, **Denoising Diffusion Probabilistic Model** | 잡음 예측을 다음 상태로 바꾸는 반복 / turn a noise prediction into the next state | §3–§5의 표집기 |
| LDM, **Latent Diffusion Model** | 압축된 표현에서 확산을 수행한다 / diffuse in a compressed representation | 강의에서 설명; 여기서는 구현하지 않음 |

**왜 다음 방법이 필요한가 / Why add another method?** 조건 입력만으로도 요청한 클래스의 분포를 배우고 표집할 수 있습니다. 실제 클래스가 겹친다면 좋은 조건부 모형도 그 겹침을 남깁니다. 따라서 CFG는 그 겹침을 무조건 고치는 방법이 아닙니다. **분포를 충실하게 재현하는 것과 요청한 특징을 더 강조하는 것은 다른 목적**이며, 후자의 조절 장치가 필요할 때 guidance를 선택합니다. 강조한 뒤에는 퍼짐·이동·실제 쓰임을 다시 평가해야 합니다.
*Conditioning already supports sampling from the requested class. A faithful model can retain real overlap. CFG is not an automatic repair for that overlap: reproducing the distribution and emphasizing a requested property are different goals. Guidance supplies an extra control when that emphasis is wanted, and its effects must be evaluated.*

**훈련 → 표집 → 평가.** 라벨을 가끔 가리는 학습으로 두 예측을 준비하고, 표집 때 CFG를 적용한 뒤, 같은 난수로 얻은 점의 분포를 비교합니다. 오늘의 2차원 점은 이미지 잠재벡터가 아닙니다. 새로운 기능을 배운다는 것이 큰 guidance scale이 언제나 좋은 결과라는 뜻은 아닙니다.
*Train the two prediction modes, apply CFG while sampling, then compare the generated distributions using the same random numbers. These 2-D points are not encoded images. Learning an extra control does not mean that stronger guidance is always better.*

**자료 연결 / Companion deck (36 slides):** `Week12_Conditional Diffusion & Guidance_v23.pptx`. 슬라이드 제목을 기준으로 연결하며, 마지막 매칭 표에서 현재 번호를 확인합니다.

빈칸은 **하나**(§4, CFG 한 줄)입니다. 나머지 코드는 설명과 함께 주어집니다. 각 절의 **예측 칸**에 먼저 답을 적고, 실행한 뒤, 맞았는지 보세요.
*There is **one** blank (§4, the CFG line). Everything else is given, with explanations. Write your guess in each section's **prediction cell** first, then run, then check.*

> 시작하기 전에 **파일 > Drive 에 사본 저장** 을 하세요. 이 링크는 읽기 전용으로 열립니다.
> *Before you start: **File > Save a copy in Drive**. This link opens read-only.*

**v11 실행 안내 / Running v11.** CPU로 실행하는 비채점 연습입니다. §4의 한 줄을 채운 뒤 그 셀의 자가검증을 먼저 통과하세요. 직접 채웠다면 다음 정답 셀은 건너뛰세요. 접힌 정답은 열어 볼 수 있으며 숨겨진 평가용 답안이 아닙니다.
*This ungraded practice uses the CPU. Fill §4's one line and pass its self-check before running any solution. If you filled it yourself, skip the next solution cell. The collapsed solution is accessible practice material.*


**수치의 출처 / Provenance.** 덱의 기존 그림·수치는 2026-11-01 v5 실험에서 이어 온 기본 실험입니다. v9는 데이터·시드·망·학습·표집 설정을 그대로 두고, v23의 수식 설명과 별도 숫자 예제를 맞췄습니다. 교수용 실행본에는 이번 v9를 로컬의 새 커널에서 실행한 출력을 저장합니다. 실제 Google Colab 실행은 확인하지 않았습니다. 환경이 바뀌면 시드를 고정해도 수치가 달라질 수 있습니다.
*The original deck experiment is retained. v11 preserves its data, seeds, network, training and sampling settings and adds arithmetic matched to v23. The instructor copy contains a fresh local-kernel run of v11. Actual Google Colab execution is unverified; fixed seeds do not guarantee equality across environments.*

**실행과 수업 활동 / Runtime and class activity.** 실행 환경은 맨 마지막 셀에, 실측 시간은 별도 검증 문서에 기록합니다. 실행 시간은 학생이 그림을 해석하고 빈칸을 푸는 시간을 포함하지 않습니다. 기존 실습 활동 약 36분을 기준으로 하되, 아래 참고 계산은 수업 진행에 맞춰 건너뛸 수 있습니다.
*Measured execution time is separate from learning time. The existing activity plan is about 36 minutes; optional reference arithmetic is not an additional assessed task.*


## 먼저. 기호와 두 종류의 반복 / First: symbols and two kinds of loops
슬라이드 「Which Noise Is Known, and Which Is Predicted?」 와 「Week 9 in Three Lines」 의 기호입니다. 오늘의 점은 **숫자 두 개로 된 가상 자료 한 개**입니다. 이미지의 원리를 작은 문제로 보는 것이지, 실제 이미지를 줄인 좌표가 아닙니다.
*The symbols on "Which Noise Is Known, and Which Is Predicted?" and "Week 9 in Three Lines". Today one data point is **one made-up observation with two numbers**: a small version of the image problem, not the coordinates of a real compressed image.*

| 기호 / Symbol | 말로 / In words | 이 실습에서 / In this lab |
|---|---|---|
| $x_0$ | 잡음을 넣기 전의 원본<br>the clean data | 숫자 두 개로 된 점<br>a point with two numbers |
| $x_t$ | 잡음 수준 t 의 중간 상태<br>the data at noise level t | 원본과 잡음이 섞인 점<br>the point mixed with noise |
| $\varepsilon$ | 우리가 실제로 섞은 잡음<br>the noise we actually added | 학습 때 아는 **정답**<br>the **target**, known in training |
| $\varepsilon_\theta(x_t, t, y)$ | 신경망이 예측한 잡음, θ 는 망의 가중치<br>the noise the network predicts; θ = its weights | 정답 ε 와 비교하는 값<br>compared with the target ε |
| $\hat\varepsilon$, $\hat x_0$ | 모자(hat)는 추정값<br>a hat marks an estimate | CFG 로 합친 잡음 예측, 그것으로 짐작한 원본<br>the guided noise prediction, and the clean point guessed from it |
| $y$, $w$ | 요청한 클래스, 그 요청을 얼마나 세게 반영할지<br>the requested class, and how strongly to follow it | 클래스 번호, 안내 세기<br>the class index, the guidance scale |

**학습과 생성은 다른 반복입니다.** 학습(§2)의 `step = 1 … 3000` 에서는 **가중치**를 고칩니다. 점마다 잡음 수준 t 를 1 ~ 100 에서 뽑습니다. 생성(§3 ~ §5)의 `t = 100 → 1` 에서는 **가중치를 고정하고 점의 좌표만** 고칩니다. 생성 한 스텝에 망을 두 번 부르는 것(§4)은 망 두 개를 따로 학습한다는 뜻이 아닙니다.
***Training and generating are different loops.** Training (§2) runs `step = 1 … 3000` and changes the **weights**, drawing a noise level t from 1 to 100 for each point. Generating (§3 to §5) runs `t = 100 → 1` with the **weights fixed** and changes only the points' coordinates. Calling the network twice per generating step (§4) does not mean two separately trained networks.*

**10초 확인 / Ten-second check.** 학습 손실의 정답은 ① 우리가 섞은 잡음 ε ② 클래스 번호 y 중 무엇인가요? 생성하는 동안 바뀌는 것은 ① 망의 가중치 ② 점의 좌표 중 무엇인가요?
***Ten-second check.** Is the training target ① the noise ε we added, or ② the class index y? While generating, what changes: ① the network's weights, or ② the points' coordinates?*

<details><summary>답 보기 / Show the answer</summary>

학습의 정답은 ε, 생성하는 동안 바뀌는 것은 점의 좌표입니다. y 는 맞히는 대상이 아니라 망에 주는 조건입니다.
*The training target is ε; while generating, the points' coordinates change. y is not something to predict: it is the condition given to the network.*
</details>

**추가 기호 / More notation.** $\varepsilon_u$: null을 넣은 예측; $\varepsilon_c$: 요청한 class를 넣은 예측; $\bar\alpha_t$: 남은 신호의 분산 계수; $\sqrt{\bar\alpha_t}$: 실제로 원본에 곱하는 계수. 문자 $u$는 unconditional, $c$는 conditional입니다. DDIM = **Denoising Diffusion Implicit Models**는 별도 표집 규칙이며, 이번 코드의 DDPM 잡음 항만 삭제한 것이 아닙니다.


**v23 연결:** 4–5장은 “학습 때 아는 잡음”을, 6–7장은 “망에 알려 주는 요청”을 구분합니다. 먼저 아래 작은 계산으로 **잡음을 만든다 → 예측과 정답을 비교한다**를 확인한 뒤, 같은 망에 요청을 추가합니다. $\theta$는 고치는 대상, $\varepsilon$는 맞히는 정답, $y$는 참고하도록 주는 조건입니다.
*Slides 4–7: the weights are updated, noise is the target, and the label is the condition. They have different jobs.*


## §0 준비 / Setup
아래 셀을 실행하면 라이브러리를 불러오고, 9주차와 같은 **잡음 요리법**(스텝마다 넣는 잡음의 분산 $\beta_t$, $\alpha_t=1-\beta_t$의 누적곱 $\bar\alpha_t$)을 만듭니다. 외부 데이터 다운로드는 없습니다. NumPy·PyTorch·Matplotlib가 있는 환경을 사용합니다. **쉬는 시간에 실행해 두세요.**
*Run the cell below. It imports the libraries and builds the same **noise recipe** as Week 9 (the per-step noise variance $\beta_t$, and the product $\bar\alpha_t$ of $\alpha_t=1-\beta_t$). No external dataset is downloaded. NumPy, PyTorch and Matplotlib must be available. **Run it during the break.***

**오늘의 T = 100 / Today's T = 100.** 일반 설명에서 쓰는 1,000스텝과 이 실습의 100스텝은 서로 다른 설정입니다. 이 스케줄에서는 $\bar\alpha_{100}\approx0.01572$, $\sqrt{\bar\alpha_{100}}\approx0.12538$이므로 원본 신호 계수가 0은 아닙니다. $x_T\sim\mathcal N(0,I)$에서 시작하는 표집은 근사입니다. 충분히 noising하면 신호가 작아지고 분포는 Gaussian noise에 가까워지지만, 여기서 정확히 같은 분포라고 단정하지 않습니다.
*The 1,000-step illustration and this 100-step lab are different settings. Here the terminal signal coefficient is about 0.12538, not zero. Starting sampling from $\mathcal N(0,I)$ approximates the terminal forward distribution; enough noising makes the signal small and approaches Gaussian noise.*

**이 절의 기호 / Notation.** $T$: 총 표집 스텝 수; $t$: 현재 잡음 수준; $\beta_t$: forward 한 스텝의 잡음 분산; $\alpha_t=1-\beta_t$; $\bar\alpha_t=\prod_{j=1}^{t}\alpha_j$. 누적 **분산 계수**와 그 제곱근인 **신호 계수**를 구분합니다.

**세 종류의 seed / Three kinds of seed.** `DATA_SEED=29`는 학습 점, `TRAIN_SEED=29`는 망 초기값과 학습 중 무작위 선택, `SAMPLE_SEED=101`은 표집 잡음을 고정합니다. 아래 기본값은 원본 실험과 같습니다.

| 바꿀 것 / Change | 다시 실행할 범위 / Rerun | 확인하는 질문 / Question |
|---|---|---|
| `SAMPLE_SEED` 또는 요청 클래스 / sampling seed or requested class | §3–§5, 같은 학습망 유지 | 같은 망이 다른 잡음·요청에서도 어떤 표본을 만드나? |
| `TRAIN_SEED` | §2–§5, 망을 처음부터 재학습 | 초기화·학습 무작위성에 결과가 민감한가? |
| `DATA_SEED`, `P_DROP`, 잡음 스케줄 / data, label-drop probability, schedule | 변경 위치부터 끝까지; 반드시 재학습 | 다른 학습 자료·설정에서 결과가 유지되나? |

seed·요청 클래스·잡음 스케줄은 §0에서, `P_DROP`은 §2에서 바꿉니다. 변경한 셀을 실행한 뒤 위 표의 범위를 다시 실행합니다. 이미 배운 클래스 0·1·2 중 요청을 바꾸는 것은 재학습을 요구하지 않습니다. 학습하지 않은 새로운 클래스나 입력 표현을 추가하는 것은 다른 문제입니다. **sampling seed를 여러 번 바꾸는 것은 여러 번 재학습하는 실험이 아닙니다.**
*Change seeds, the requested class and the noise schedule in §0; change `P_DROP` in §2. Run the edited cell, then rerun the range in the table. Choosing among already trained classes does not require retraining; adding an unseen class or a new input representation is different. Several sampling seeds from one trained model are not several independent training runs.*


In [ ]:
import math, time, copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.set_num_threads(2)   # CPU 스레드 고정; 환경 간 완전 동일성을 보장하지 않음 / fixed CPU threads; not a cross-environment reproducibility guarantee

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
# Message shown when a TODO is still blank. It means the notebook is fine, not broken.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막힐 때 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace None above with your code and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell just below and continue.\n"
)

GREEN, GOLD, RED, GRAY = "#406551", "#C9971C", "#B04A3A", "#9AA5A0"   # 그림 색 / figure colors

# 잡음 요리법(9주차와 같음). t = 1 … T 가 스텝 번호, 0 번 칸은 「잡음을 더하기 전」(x_0 = 데이터).
# The noise recipe (same as Week 9). t = 1 … T are the step numbers; slot 0 stands for "before any noise" (x_0 = the data).
DATA_SEED = 29
TRAIN_SEED = 29
SAMPLE_SEED = 101
TARGET = 2  # choose 0, 1 or 2; all three were present in training
T = 100
betas = torch.cat([torch.zeros(1), torch.linspace(0.0008, 0.08, T)])   # β_0 = 0, β_1 … β_T 가 직선으로 커진다 / β_0 = 0, β_1 … β_T rise linearly
alphas = 1 - betas                                                      # α_t = 1 − β_t
abar = torch.cumprod(alphas, 0)                                         # ᾱ_t = α_1 × … × α_t. ᾱ_0 = 1 / the running product; ᾱ_0 = 1

print("t   β_t      ᾱ_t      남는 신호 √ᾱ_t / surviving signal")
for t in (1, 25, 50, 100):
    print(f"{t:3d} {betas[t]:.4f}  {abar[t]:.4f}  {abar[t].sqrt():.3f}")
# 기대값 / Expected   (9주차 노트북과 같은 표 / the same table as the Week 9 notebook)
#     1 0.0008  0.9992  1.000
#    25 0.0200  0.7697  0.877
#    50 0.0400  0.3556  0.596
#   100 0.0800  0.0157  0.125

print(f"Terminal: alpha_bar_T={abar[T].item():.8f}, signal={abar[T].sqrt().item():.8f}, noise={(1-abar[T]).sqrt().item():.8f}")


### 몸풀기 0. 왜 잡음을 맞히는가? / Why predict the noise? · v23 4–5장
잡음을 없애는 망을 가르치려면 **무엇을 정답으로 삼을지** 알아야 합니다. 학습에서는 우리가 깨끗한 점에 잡음을 직접 섞으므로, 넣은 잡음 $\varepsilon$을 정확히 알고 있습니다. 망에는 섞인 값 $x_t$와 잡음 수준 $t$를 주고 그 잡음을 예측하게 합니다. 생성 때에는 원본을 모르므로 이 학습된 예측을 사용합니다.

$$x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon$$

| 읽는 순서 | 이 예에서 | 뜻 |
|---|---|---|
| 남긴 원본 | $0.8\times0.5=0.4$ | 신호 계수 × 깨끗한 원본 |
| 섞은 잡음 | $0.6\times1=0.6$ | 잡음 계수 × 뽑은 잡음 |
| 망이 볼 입력 | $0.4+0.6=1.0$ | 두 항을 더한 $x_t$ |

**기호:** $x_0$: 원본; $x_t$: 잡음 섞인 입력; $\bar\alpha_t=0.64$: 남은 신호의 분산 계수; $\sqrt{\bar\alpha_t}=0.8$: 신호 계수; $\sqrt{1-\bar\alpha_t}=0.6$: 잡음 계수. 두 계수 자체의 합이 1인 것은 아닙니다. 제곱한 값이 $0.64+0.36=1$입니다.

다음 **별도 손실 예제**에서는 정답 잡음이 $0.5$, 예측 잡음이 $0.2$입니다. 오차는 $0.5-0.2=0.3$, 제곱오차는 $0.3^2=0.09$입니다. 여러 점·좌표에서 이를 평균한 것이 MSE(Mean Squared Error, 평균제곱오차)입니다. **이 식이 필요한 이유는 신경망의 예측을 얼마나 고칠지 정할 기준을 만들기 위해서**입니다. 앞의 forward 예제에서 잡음 1을 넣었다는 것과 이 별도 예제를 섞지 마세요.
*The forward example uses noise 1. The separate loss example uses target noise 0.5 and predicted noise 0.2. Squared error is 0.09. Average it across samples and coordinates to obtain MSE; training changes the weights to reduce this discrepancy.*

**손실 기호:** $L$은 손실, $\mathbb E[\cdot]$는 여러 학습 경우에 대한 평균, $\|\cdot\|^2$는 좌표별 차이를 제곱해 더하는 표기입니다. 본 코드의 `.square().mean()`은 미니배치와 두 좌표에 대해 평균합니다. 아래는 제공된 계산이며 새 빈칸이 아닙니다.

In [ ]:
# Supplied toy arithmetic. No random draws; separate from the training experiment.
toy_forward_x0, toy_forward_eps, toy_forward_abar = 0.5, 1.0, 0.64
toy_forward_signal = math.sqrt(toy_forward_abar)
toy_forward_noise = math.sqrt(1 - toy_forward_abar)
toy_forward_xt = toy_forward_signal * toy_forward_x0 + toy_forward_noise * toy_forward_eps
print(f"Forward: 0.8 × 0.5 + 0.6 × 1 = {toy_forward_xt:.1f}")

toy_target_eps, toy_predicted_eps = 0.5, 0.2  # separate loss example
toy_error = toy_target_eps - toy_predicted_eps
toy_squared_error = toy_error ** 2
print(f"Loss example: difference {toy_error:.1f}; squared error {toy_squared_error:.2f}")
assert math.isclose(toy_forward_xt, 1.0)
assert math.isclose(toy_squared_error, 0.09)


## 몸풀기. 슬라이드의 숫자 / Warm-up. The numbers from the slides
「CFG on One Number」 와 「What the Sampler Does With the Guided Noise」 의 숫자를 그대로 씁니다. 조건 없는 예측 $\varepsilon_u = 0.2$, 조건 있는 예측 $\varepsilon_c = 0.5$, 그리고 $x_t = 1.0$, 신호 계수 0.8, 잡음 계수 0.6. 빈칸은 없습니다. 숫자가 슬라이드와 같은지만 보세요.
*The numbers from those two slides, as they are: the unconditional prediction $\varepsilon_u = 0.2$, the conditional one $\varepsilon_c = 0.5$, and $x_t = 1.0$ with signal weight 0.8 and noise weight 0.6. No blank here. Just check that the numbers match the slides.*

이 숫자는 설명용이며 실제 100스텝 스케줄의 특정 스텝을 지정하지 않습니다. 이 노트북의 convention은 $\hat\varepsilon=\varepsilon_u+w(\varepsilon_c-\varepsilon_u)$입니다. **w = 0: unconditional, w = 1: ordinary conditional, w > 1: extrapolation.** 7.5는 큰 scale의 예시일 뿐 권장 기본값이 아닙니다.
*These are illustrative numbers, not a specified step of the 100-step schedule. In this convention, w = 0 is unconditional, w = 1 ordinary conditional, and w > 1 extrapolation. The value 7.5 is an example of a large scale, not a recommended default.*

**숫자에서 읽을 것 / Read the numbers.** $\varepsilon_c-\varepsilon_u=0.3$은 조건이 잡음 예측에 더한 차이입니다. 먼저 $w=1$일 때 0.5가 돌아오는지 확인한 뒤 $w=2$에서 0.8로 연장되는 것을 봅니다. 일반식은 이 계산을 모든 좌표에 적용한 것입니다. $\hat x_0$는 그 예측으로 계산한 원본 추정값이며, 다음 표집 상태와 같지 않습니다. 참고 절에서 둘을 구분하는 한 스텝 계산을 볼 수 있습니다.


**v23 10–13장: 왜 빼고 나누나?** 먼저 조건이 예측을 바꾼 양은 $0.5-0.2=0.3$입니다. 이것을 두 배 강조하면 guided noise는 $0.2+2\times0.3=0.8$입니다. 이제 현재 값 1 안에 섞였다고 예측한 잡음의 기여를 계산합니다.

| 순서 | 계산 | 계산하는 이유 |
|---|---|---|
| ① 잡음의 기여 | $0.6\times0.8=0.48$ | 예측한 잡음에 forward 잡음 계수를 곱한다 |
| ② 남은 신호 | $1-0.48=0.52$ | 현재 값에서 잡음의 기여를 뺀다 |
| ③ 원본 추정 | $0.52/0.8=0.650$ | forward에서 원본을 0.8배 했으므로 다시 나눈다 |

**기호를 다시 읽기:** $\varepsilon_u=0.2$: 조건 없는 예측; $\varepsilon_c=0.5$: 조건 있는 예측; $w=2$: 차이를 강조하는 배수; $\hat\varepsilon=0.8$: 합친 잡음 예측; $x_t=1$: 현재 값; $\hat x_0$: 추정한 원본. 여기서 두 번 나오는 **0.8은 서로 다른 양**입니다. 하나는 guided noise, 다른 하나는 신호 계수이며 우연히 숫자만 같습니다. $\hat x_0$는 정답 원본이나 다음 상태를 바로 뜻하지 않습니다.
*The chain is noise contribution 0.48 → remaining signal 0.52 → clean estimate 0.650. The two values 0.8 have different roles: predicted noise and signal coefficient.*


In [ ]:
# 다음 이름을 씁니다 / Names used below
#   eps_u   : 조건 없는 잡음 예측 ε_u / the unconditional noise prediction ε_u
#   eps_c   : 조건 있는 잡음 예측 ε_c / the conditional noise prediction ε_c
#   w       : 안내 세기 w / the guidance scale w
#   eps_hat : 안내한 예측 ε̂ = ε_u + w (ε_c − ε_u) / the guided prediction
#   x0_hat  : ε̂ 로 짐작한 깨끗한 값 x̂_0 = (x_t − 0.6 ε̂) / 0.8 / the clean estimate from ε̂

eps_u, eps_c = 0.2, 0.5
print(" w    ε̂ = 0.2 + w × 0.3     x̂_0 = (1.0 − 0.6 ε̂) / 0.8")
for w in (0, 1, 2, 7.5):
    eps_hat = eps_u + w * (eps_c - eps_u)              # 「The CFG Rule」 의 한 줄 / the one line of that slide
    x0_hat = (1.0 - 0.6 * eps_hat) / 0.8               # 「What the Sampler Does With the Guided Noise」 / the clean estimate on that slide
    print(f"{w:4}  {eps_hat:6.2f}                 {x0_hat:7.4f}")
# 기대값 / Expected
#    w    ε̂                       x̂_0
#    0    0.20                     1.1000
#    1    0.50                     0.8750
#    2    0.80                     0.6500
#    7.5  2.45                    -0.5875
#   w 가 커질수록 ε̂ 는 커지고 x̂_0 은 작아집니다. 안내한 잡음은 그림에 더하는 것이 아니라 그림에서 빼는 것입니다.
#   As w grows, ε̂ grows and x̂_0 falls: the guided noise is subtracted from the picture, not added to it.

# Slide 13: name each intermediate quantity rather than hiding it in one line.
toy_guided_noise = 0.2 + 2 * (0.5 - 0.2)
toy_noise_contribution = 0.6 * toy_guided_noise
toy_remaining_signal = 1.0 - toy_noise_contribution
toy_clean_estimate = toy_remaining_signal / 0.8
print(f"Noise contribution {toy_noise_contribution:.2f} -> remaining signal {toy_remaining_signal:.2f} -> clean estimate {toy_clean_estimate:.3f}")
assert math.isclose(toy_noise_contribution, .48)
assert math.isclose(toy_remaining_signal, .52)
assert math.isclose(toy_clean_estimate, .65)


**부호 확인 / Sign check.** 좌표가 둘이고 $\varepsilon_u = [0.2,\ 0.5]$, $\varepsilon_c = [0.5,\ 0.2]$ 라면 차이는 $[+0.3,\ -0.3]$ 입니다. w = 2 일 때 어느 좌표가 커지고 어느 좌표가 작아질까요?
***Sign check.** With two coordinates, $\varepsilon_u = [0.2,\ 0.5]$ and $\varepsilon_c = [0.5,\ 0.2]$, the difference is $[+0.3,\ -0.3]$. At w = 2, which coordinate grows and which shrinks?*

<details><summary>계산한 뒤 답 보기 / Show the answer after you compute</summary>

안내한 예측은 $[0.2 + 2 \times 0.3,\ 0.5 - 2 \times 0.3] = [0.8,\ -0.1]$ 입니다. 첫 좌표는 커지고 둘째는 작아집니다. CFG 는 모든 값을 키우는 것이 아니라, 좌표마다 조건이 바꾼 방향을 키웁니다. §4 의 자가검증도 이 예를 씁니다.
*The guided prediction is $[0.2 + 2 \times 0.3,\ 0.5 - 2 \times 0.3] = [0.8,\ -0.1]$: the first coordinate grows, the second shrinks. CFG does not make every value bigger; in each coordinate it amplifies the direction the condition changes. §4's self-check uses this example too.*
</details>

## §1 세 클래스가 겹치는 자료 / Three classes that overlap
오늘의 데이터는 2차원 점 3,000개, 클래스 셋(0 · 1 · 2)입니다. 세 중심은 반지름 1.5 인 원 위에 120° 간격으로 있고, 각 클래스는 중심 둘레에 표준편차 0.8 로 흩어져 **서로 겹칩니다.** 겹치게 만든 이유가 오늘의 핵심입니다. 클래스 번호만으로는 점이 어디에 놓일지 **일부만** 정해집니다. 글로 이미지를 주문할 때와 같은 사정입니다. 「고양이 사진」이라는 조건은 사진을 일부만 정합니다.
*Today's data are 3,000 points in 2-D with three classes (0, 1, 2). The three centers sit 120° apart on a circle of radius 1.5, and each class scatters around its center with standard deviation 0.8, so the classes **overlap**. The overlap is the point of today: the class index pins the point down only **partly**, just as a text prompt like "a photo of a cat" pins a picture down only partly.*

**예측 1.** 클래스 2 를 주문하면 완벽한 조건부 모형의 점은 전부 클래스 2 의 영역에 놓일까요? 아니면 일부가 이웃 클래스 쪽에도 떨어질까요?
***Prediction 1.** If we ask for class 2, will a perfect conditional model put every point inside class 2's region, or will some land toward the neighbors?*

**그림 읽는 법 / How to read the picture.** 점 하나가 자료 하나이고, 가로축은 첫째 좌표, 세로축은 둘째 좌표입니다. 픽셀 밝기도 시간도 아닙니다. `+` 와 번호는 세 클래스의 중심이고, 오른쪽 아래의 2 가 오늘 주문할 목표입니다. 클래스 2 는 손글씨 숫자 2 의 그림이 아닙니다. 이 첫 그림의 **색은 원래 클래스 번호**입니다. 원래 클래스가 2 인 점도 중심 1 에 더 가까울 수 있습니다. **원래 라벨과 가장 가까운 중심은 다른 정보입니다.** 덱 「What Do the Dots in Today’s Pictures Show?」 의 그림이 이 셀입니다.
*One dot is one observation: the horizontal axis is its first coordinate, the vertical axis its second. Neither is a pixel brightness or time. `+` and the numbers mark the three class centers; the 2 at the lower right is today's target. Class 2 is not a picture of a handwritten 2. In this first picture **color is the original class label**. A point whose label is 2 can still lie nearer center 1: **the original label and the nearest center are different information.** The figure on "What Do the Dots in Today’s Pictures Show?" is this cell.*

**이 절의 기호 / Notation.** `K=3`: 데이터 클래스 수; `NULL=3`: 네 번째 입력 자리(네 번째 데이터 클래스가 아님); `CENTERS[k]`: 클래스 k의 진짜 중심; `SPREAD=0.8`: 각 좌표의 모집단 표준편차. 데이터의 라벨과 최근접 중심 번호는 구분합니다.


In [ ]:
# 예측 1 / Prediction 1. 빈칸 ___ 을 내 예상으로 채우세요. 맞힐 필요는 없고, 실행한 뒤 비교합니다. 이 셀은 실행하지 않아도 됩니다.
# Fill the ___ blanks with your guess. You do not need to be right; compare after running. This cell does not need to be run.
my_prediction_1 = "클래스 2 를 그대로 베끼는 모형이라면 이웃 중심 쪽의 점도 ___ 만든다. 그래서 중심 2 에 가장 가까운 점은 약 ___ % 일 것이다."
my_prediction_1_en = "A model that copies class 2 faithfully will ___ make points near the neighboring centers, so about ___ % will be nearest to center 2."   # 영어로 적고 싶으면 이 줄에 / or write it in English on this line

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   CENTERS : 세 클래스의 중심, 크기 (3, 2) / the three class centers, shape (3, 2)
#   SPREAD  : 중심 둘레의 흩어짐(표준편차) / the scatter around a center (standard deviation)
#   K       : 클래스 수 3. 번호 3 은 「조건 없음」(null) 자리 / the number of classes, 3; index 3 is the "no condition" (null) slot
#   data    : 학습 점 3,000개, 크기 (3000, 2) / the 3,000 training points, shape (3000, 2)
#   labels  : 점마다의 클래스 번호 0 · 1 · 2, 크기 (3000,) / the class index per point, shape (3000,)

angles = torch.tensor([90., 210., 330.]) * math.pi / 180                 # 120° 간격 / 120° apart
CENTERS = torch.stack([1.5 * torch.cos(angles), 1.5 * torch.sin(angles)], 1)
SPREAD = 0.8
K, NULL = 3, 3

def make_data(n, gen):
    """클래스 번호를 고르고 그 중심 둘레에 흩뿌린 점 n 개 / n points: pick a class index, scatter around its center."""
    y = torch.randint(0, K, (n,), generator=gen)                          # 점마다 클래스 번호 / a class index per point
    return CENTERS[y] + torch.randn(n, 2, generator=gen) * SPREAD, y        # 중심 + 흩어짐, 그리고 번호 / center plus scatter, and the index

data, labels = make_data(3000, torch.Generator().manual_seed(DATA_SEED))         # 데이터 난수 고정 / fix the data's randomness

def nearest_center(pts):
    """점마다 가장 가까운 중심의 번호 / the index of the nearest center, per point."""
    return torch.cdist(pts, CENTERS).argmin(1)

fig, ax = plt.subplots(figsize=(3.6, 3.4), dpi=130)
for k, col in zip(range(K), (GRAY, GOLD, GREEN)):
    pts = data[labels == k]
    ax.scatter(pts[:, 0], pts[:, 1], s=3, color=col, alpha=0.5, label=f"class {k}")
ax.scatter(CENTERS[:, 0], CENTERS[:, 1], marker="+", s=80, color="black", linewidths=1.5)
for k, center in enumerate(CENTERS):                                         # 중심마다 번호, 목표는 2 / a number at each center; the target is 2
    ax.annotate(f"{k}" + (" (target)" if k == TARGET else ""), center.tolist(), xytext=(5, 6), textcoords="offset points", fontsize=9, weight="bold")
ax.set(xlim=(-4, 4), ylim=(-4, 4), xticks=[-2, 0, 2], yticks=[-2, 0, 2], xlabel="coordinate 1", ylabel="coordinate 2",
       title="Three overlapping classes (+ = centers)"); ax.set_aspect("equal")
ax.legend(fontsize=8, loc="lower left", frameon=False, markerscale=4); plt.tight_layout(); plt.show()

print("클래스 / class   점 수 / points   가장 가까운 중심이 자기 중심인 비율 / share whose nearest center is its own")
own = nearest_center(data) == labels
for k in range(K):
    print(f"{k:5d}            {(labels == k).sum().item():5d}           {100 * own[labels == k].float().mean():.0f} %")
# 기대값 / Expected
#   클래스마다 약 1,000점. 자기 중심이 가장 가까운 점은 약 90 % 뿐입니다. 나머지 10 % 는 이웃 중심 쪽에 더 가깝습니다. 이것이 「겹침」 입니다.
#   About 1,000 points per class. Only about 90 % of a class's points are nearest to their own center; the other 10 % lie closer to a neighbor. That is the overlap.

## §2 조건을 받는 잡음 예측기, 그리고 한 줄 (제공) / A noise predictor that takes the condition, plus one line (provided)
9주차 §3 의 신경망에 입력 하나가 늘었습니다. 클래스 번호 y 를 네 칸짜리 원핫으로 펴서(0 · 1 · 2, 그리고 **「조건 없음」 null 칸**) 좌표와 t 옆에 붙입니다. 손실은 9주차 그대로, 섞은 잡음을 맞히는 제곱 오차입니다.
*The Week 9 network gains one input. The class index y is spread into a four-slot one-hot (0, 1, 2, and a **"no condition" null slot**) and appended next to the coordinates and t. The loss is unchanged: squared error on the mixed-in noise.*

새 줄은 하나입니다. **점마다** 확률 0.15 로 라벨을 null 로 바꿉니다(「Training Changes by One Line」). 그래서 한 신경망이 조건 있는 예측 $\varepsilon_c$ 와 조건 없는 예측 $\varepsilon_u$ 를 **둘 다** 배웁니다. 이 한 망 설계에서 null 입력을 학습하지 않으면, §4에 필요한 조건 없는 예측 분기가 준비되지 않습니다.
*The one new line: with probability 0.15, **per point**, the label is replaced by null (slide 「Training Changes by One Line」). So one network learns **both** the conditional prediction $\varepsilon_c$ and the unconditional one $\varepsilon_u$. In this shared-network design, omitting training on null inputs leaves the unconditional branch needed in §4 untrained.*

**입력 하나가 어떻게 바뀌나 / A small input example.** 슬라이드19와 같은 실제 라벨 `[2, 1, 0, 2]`를 씁니다. 이번에 뽑힌 dropout 표시가 `[False, True, False, False]`라면 망에 넣는 라벨은 `[2, 3, 0, 2]`입니다. 슬라이드의 NULL이 코드에서는 숫자3으로 표시됩니다. 원본 점과 정답 잡음은 지우지 않고, 두 번째 점의 **라벨 정보만** 가립니다. 0.15는 매 점에서 null로 바꿀 확률이며, 모든 미니배치에서 정확히 15%를 바꾸라는 뜻은 아닙니다.
*Use slide 17's labels [2, 1, 0, 2]. For a sampled mask [False, True, False, False], the network receives [2, 3, 0, 2]. Index 3 represents the slide's NULL. Only the second point's label information is hidden; the point and target noise stay. Probability 0.15 does not require exactly 15% null labels in every minibatch.*

**이 절의 기호 / Notation.** $\theta$: 학습되는 망의 가중치; $t$: 잡음 수준; $y_{in}$: 실제로 넣는 라벨 또는 null; $\varepsilon$: 뽑아 섞은 정답 잡음; $\varepsilon_\theta$: 망의 잡음 예측; MSE = **Mean Squared Error**, 좌표별 제곱오차의 평균. `P_DROP`: null로 바꿀 확률.


**손실을 읽는 법 / Reading the loss.** 최적 제곱오차 예측은 $E[\varepsilon\mid x_t,t,y_{in}]$입니다. 같은 입력에도 가능한 잡음이 여러 개라 최적 예측기의 기대 손실도 0보다 클 수 있습니다. **하지만 원본에서 관찰한 약 0.392 전체를 irreducible error라고 해석할 수는 없습니다.** 이 값은 학습된 망의 마지막 100개 미니배치 손실 평균이며 이론적 하한을 계산한 값이 아닙니다.
*The optimal squared-error predictor is the conditional mean of the noise. Remaining conditional uncertainty can make even its expected loss nonzero. The original run's 0.392 is the trained network's mean loss over the last 100 minibatches, not a computed irreducible lower bound.*

**이 학습이 다음 절을 준비한다 / Why training comes before guidance.** §4에서 두 예측의 차이를 쓰려면 같은 망이 요청이 있을 때와 없을 때 모두 유효한 예측을 내야 합니다. 그래서 지금 null 입력을 학습합니다. 이 단계는 두 가지 입력에 반응하도록 망을 **학습**시키며, 나중에 어느 정도 차이를 강조할지는 표집 때 고르는 $w$가 정합니다.
*Before §4 can combine two predictions, the shared network must handle both requested and no-request inputs. Label dropout prepares those two modes during training. The sampling scale w later controls how strongly their difference is emphasized.*


**v23 17장의 확률 계산:** 설명용 난수 `[0.70, 0.10, 0.80, 0.40]`를 0.15와 비교하면 두 번째 값만 작습니다. 따라서 `[2,1,0,2]`가 `[2,NULL,0,2]`가 됩니다. 이 작은 배치의 실제 비율은 $1/4=25\%$이며 설정 확률 15%와 같을 필요가 없습니다. 아래 실제 학습은 각 점에서 새 난수를 뽑습니다. 설명용 난수는 고정된 숫자이며 학습 난수열에 추가하지 않습니다.


In [ ]:
# 다음 이름을 씁니다 / Names used below
#   t_features : 스텝 t 를 사인 · 코사인 16개로 펴는 함수(9주차와 같음) / spreads the step t into 16 sines and cosines (as in Week 9)
#   y_features : 클래스 번호를 원핫 4칸으로 펴는 함수. 번호 3 = null / spreads the class index into a 4-slot one-hot; index 3 = null
#   CondNet    : 조건을 받는 잡음 예측기 ε_θ(x_t, t, y). 입력 2 + 16 + 4, 은닉 64 × 2, 출력 2 / the conditional noise predictor; input 2 + 16 + 4, hidden 64 × 2, output 2
#   jump       : 9주차 §2 의 한 줄 건너뛰기 / the one-line jump from Week 9, §2
#   P_DROP     : 라벨을 null 로 바꿀 확률 / the probability of replacing the label by null
#   drop       : 점마다 「이번에는 라벨을 뺀다」 표시, 크기 (512,) / per point, "drop the label this time", shape (512,)
#   y_in       : 신경망에 실제로 넣는 번호(라벨 또는 null) / the index actually fed to the network (the label or null)
#   model      : 학습할 신경망 / the network we train
#   losses     : 스텝마다의 손실 / the loss at every update

def t_features(t):
    """t = 1 … T 를 [0, 1) 로 줄인 뒤 주파수 8개의 사인 · 코사인으로 편다 / scale t = 1 … T into [0, 1), then spread it over 8 frequencies of sine and cosine."""
    tt = (t.float()[:, None] - 1) / T
    freqs = (2 ** torch.arange(8).float())[None, :]
    return torch.cat([torch.sin(tt * freqs * math.pi), torch.cos(tt * freqs * math.pi)], 1)

def y_features(y):
    """클래스 번호 0 · 1 · 2 · 3(null) 을 원핫 4칸으로 / the class index 0, 1, 2, 3 (null) as a 4-slot one-hot."""
    return nn.functional.one_hot(y, K + 1).float()

class CondNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + 16 + K + 1, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 2))

    def forward(self, x, t, y):
        return self.net(torch.cat([x, t_features(t), y_features(y)], 1))   # 좌표 2 + t 의 16 + 조건 4 → 잡음 2 / 2 coordinates + 16 for t + 4 for y → 2 noise entries

def jump(x0, eps, abar_t):
    """9주차 §2: x_0 에서 스텝 t 의 x_t 로 한 줄에 / Week 9, §2: from x_0 to the x_t of step t in one line."""
    return torch.sqrt(abar_t) * x0 + torch.sqrt(1 - abar_t) * eps

# A supplied dropout mask illustrates the input change; it uses no random draws.
example_y = torch.tensor([2, 1, 0, 2])
example_drop = torch.tensor([False, True, False, False])
example_y_in = torch.where(example_drop, torch.full_like(example_y, NULL), example_y)
print("Label-dropout example:", example_y.tolist(), "->", example_y_in.tolist())
assert example_y_in.tolist() == [2, 3, 0, 2]

P_DROP = 0.15
gen = torch.Generator().manual_seed(TRAIN_SEED)                                  # 미니배치 · t · ε · 드롭의 난수 고정 / fix the draws of minibatch, t, ε and dropout
torch.manual_seed(TRAIN_SEED)                                                    # 신경망 초기값 고정 / fix the initial weights
model = CondNet()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
t0 = time.time()
for step in range(1, 3001):
    idx = torch.randint(0, len(data), (512,), generator=gen)              # 학습 점 512개를 무작위로 / 512 random training points
    x0, y = data[idx], labels[idx]
    t = torch.randint(1, T + 1, (len(x0),), generator=gen)              # 점마다 스텝 t 를 무작위로 / a random step t per point
    eps = torch.randn(x0.shape, generator=gen)                           # 점마다 새 잡음 / new noise per point
    drop = torch.rand(len(x0), generator=gen) < P_DROP                   # 점마다 따로 동전을 던진다 / a separate coin flip per point
    y_in = torch.where(drop, torch.full_like(y, NULL), y)                # 「Training Changes by One Line」: 가끔 라벨을 null 로 / sometimes the label becomes null
    x_t = jump(x0, eps, abar[t][:, None])
    loss = (model(x_t, t, y_in) - eps).square().mean()                   # 9주차와 같은 손실 / the same loss as Week 9
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
print(f"처음 100스텝 평균 손실 / mean loss, first 100 updates : {np.mean(losses[:100]):.3f}")
print(f"마지막 100스텝 평균 손실 / mean loss, last 100 updates : {np.mean(losses[-100:]):.3f}")
print(f"걸린 시간 / time: {time.time() - t0:.0f} s")
# 원본 실행 참고값 / Original-run reference (2026-11-01 v5; 환경 차이로 달라질 수 있음 / may differ across environments)
#   처음 100스텝 / first 100  : 0.634
#   마지막 100스텝 / last 100  : 0.392
#   시간 / time: 이 노트북을 만든 2코어 CPU 에서 약 10초 / about 10 s on the 2-core CPU used to build this notebook
#   9주차처럼 0 까지 내려가지 않습니다. 같은 x_t 를 만드는 깨끗한 점 x_0 와 잡음 ε 의 짝이 여럿이라, x_t 만 보고는 ε 을 정확히 맞힐 수 없습니다.
#   As in Week 9 it does not reach 0: many pairs of a clean point x_0 and noise ε give the same x_t, so ε cannot be read exactly off x_t.
#   최적 예측기도 (x_t, t, y_in)을 알고 남는 잡음의 불확실성 때문에 기대 손실이 0보다 클 수 있습니다.
#   Even an optimal predictor can have nonzero expected noise-prediction loss, due to uncertainty remaining after observing (x_t, t, y_in).
#   그러나 이 학습된 망의 관찰 손실 0.392가 이론적 최솟값이라는 뜻은 아닙니다. 최적화·모형·유한 표본의 영향도 남을 수 있습니다.
#   The observed 0.392 is not necessarily the theoretical minimum; optimization, model and finite-sample effects may remain.

plt.figure(figsize=(5, 2.6))
plt.plot(losses, color=GRAY, lw=0.5, alpha=0.8)
k = 50; plt.plot(np.arange(k, len(losses) + 1), np.convolve(losses, np.ones(k) / k, mode="valid"), color=GREEN, lw=1.5)   # 50스텝 이동평균 / 50-update moving average
plt.xlabel("Adam updates"); plt.ylabel("noise MSE"); plt.ylim(0.25, 0.8); plt.title("Training loss of the conditional denoiser", fontsize=10)
plt.tight_layout(); plt.show()

## §3 조건 없이, 그리고 조건을 주고 표집 (제공) / Sample without the condition, then with it (provided)
표집은 9주차 §4 그대로입니다. 잡음에서 출발해 T 스텝을 거꾸로 걷고, 스텝마다 「예측 → 예측한 잡음의 일부 빼기 → 새 잡음 더하기」. 다른 점은 하나, 신경망에 번호를 같이 넣는 것입니다. null(3) 을 넣으면 **조건 없는** 표집, 2 를 넣으면 **클래스 2 를 주문한** 표집입니다. 슬라이드 「Can We Prefer More Class-Specific Samples?」 의 그림이 이 셀의 오른쪽 칸입니다.
*Sampling is Week 9, §4 as it was: start from noise, walk the T steps backward, and at each step "predict → remove part of the predicted noise → add new noise". The one difference: the index goes into the network too. Feeding null (3) gives **unconditional** samples; feeding 2 gives samples **that asked for class 2**. The right panel of this cell is the figure on the slide 「Can We Prefer More Class-Specific Samples?」.*

**예측 2.** 클래스 2 를 주문한 600점 가운데 가장 가까운 중심이 클래스 2 인 점은 몇 % 일까요? 100 %? §1 의 자료는 약 90 % 였습니다.
***Prediction 2.** Of the 600 points that asked for class 2, what share will have class 2 as the nearest center? 100 %? For the data in §1 it was about 90 %.*

**이제 가중치는 고정입니다.** 무작위 점에서 출발해 좌표를 100 번 고치고, 스텝마다 같은 요청(null 또는 2)을 넣습니다. 두 그림은 같은 난수(시드 101)를 써서 넣은 번호 하나만 다릅니다. **이 그림부터 색의 뜻이 바뀝니다.** 초록은 「중심 2 가 가장 가까움」, 빨강은 「다른 중심이 더 가까움」 입니다. 만든 점의 원래 라벨을 안다는 뜻이 아닙니다.
***The weights are now fixed.** Start from random points, change their coordinates 100 times, and feed the same request (null or 2) at every step. Both panels use the same random numbers (seed 101), so only the index differs. **From this picture on, color means something new:** green = "center 2 is nearest", red = "another center is nearer". It does not mean we know a generated point's original label.*

**clean estimate 다음은? / What follows the clean estimate?** 현재 예측 $\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\hat\varepsilon)/\sqrt{\bar\alpha_t}$는 원본 추정값이며 $x_{t-1}$이나 완성된 표본이 아닙니다. 아래 `ddpm_mean`은 같은 예측으로 DDPM 평균을 계산합니다.

$$\mu_\theta(x_t,t)=\frac{x_t-\beta_t\hat\varepsilon/\sqrt{1-\bar\alpha_t}}{\sqrt{\alpha_t}},\qquad x_{t-1}=\mu_\theta(x_t,t)+\sqrt{\beta_t}\,z,\quad z\sim\mathcal N(0,I).$$

이 구현은 $t>1$에서 새 독립 잡음을 더하고 $t=1$에서는 평균만 반환합니다. 따라서 흐름은 **two predictions → guided noise → clean estimate에 대응하는 DDPM mean → fresh noise → next noisy state**입니다. 코드에서는 $\hat x_0$를 따로 저장하지 않고 동등한 평균 식에 $\hat\varepsilon$를 바로 넣습니다. DDPM의 잡음 항만 지운 것을 DDIM이라고 부르지 않습니다. DDIM은 별도의 update rule을 사용하며 여기서는 구현하지 않습니다.
*The clean estimate is an estimate of the original data, not the next state or a finished sample. This code uses the DDPM mean above and fresh noise with standard deviation $\sqrt{\beta_t}$ for t > 1, and returns only the mean at t = 1. It substitutes the guided noise directly into an equivalent mean formula. Removing this noise term alone is not a DDIM implementation; DDIM uses its own update rule.*

**이 절의 기호 / Notation.** $\hat\varepsilon$: 이번 스텝에 사용할 잡음 예측; $\hat x_0$: 깨끗한 원본 추정; $\mu_\theta$: 다음 상태의 평균; $z$: 이번 스텝에서 새로 뽑는 독립 표준정규 잡음; $x_{t-1}$: 잡음까지 더한 다음 상태. $\theta$는 표집 동안 고정합니다. 여기의 $z$는 이후 LDM 슬라이드에서 쓰는 이미지 잠재표현 기호와 역할이 다릅니다.

**평균과 표준편차를 구분 / Separate the mean from the sampling standard deviation.** `ddpm_mean`은 $\hat x_0$를 쓰는 평균식과 동등합니다. 새 잡음에 곱하는 표준편차는 **이 실습에서 선택한 $\sqrt{\beta_t}$**입니다. 이 값을 정확한 forward posterior의 표준편차라고 부르지 않습니다. 평균의 동등성을 확인했다고 표집 분산의 선택까지 같아지는 것은 아닙니다.
*The mean is equivalent to the expression using the clean estimate. This implementation chooses $\sqrt{\beta_t}$ as its sampling standard deviation. Do not identify that choice with the exact forward-posterior standard deviation: equivalence of the mean does not determine the sampling variance.*


In [ ]:
# 예측 2 / Prediction 2. 빈칸 ___ 을 내 예상으로 채우세요. 맞힐 필요는 없고, 실행한 뒤 비교합니다. 이 셀은 실행하지 않아도 됩니다.
# Fill the ___ blanks with your guess. You do not need to be right; compare after running. This cell does not need to be run.
my_prediction_2 = "조건 없이 뽑으면 약 ___ %, 클래스 2 를 주문하면 약 ___ % 가 중심 2 에 가장 가까울 것이다(자료의 클래스 2 는 91 %)."
my_prediction_2_en = "Without the condition about ___ % will be nearest to center 2; asking for class 2, about ___ % (the data's class 2: 91 %)."   # 영어로 적고 싶으면 이 줄에 / or write it in English on this line

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   ddpm_mean    : 9주차 §4 의 역방향 한 스텝의 평균 부분 / the mean part of one reverse step, Week 9 §4
#   sample_plain : 번호 y 하나를 정해 놓고 T 스텝을 거꾸로 걷는 표집 / sampling that walks the T steps backward with one fixed index y
#   TARGET       : 오늘 주문하는 클래스, 2 / the class we ask for today, 2
#   share_target : 가장 가까운 중심이 클래스 2 인 점의 비율 / the share of points whose nearest center is class 2

def ddpm_mean(x_t, eps_hat, beta_t, alpha_t, abar_t):
    """「One DDPM Reverse Step」 의 평균 부분(9주차 빈칸 2) / the mean part of that slide (Week 9, blank 2)."""
    return (x_t - beta_t / torch.sqrt(1 - abar_t) * eps_hat) / torch.sqrt(alpha_t)

@torch.no_grad()
def sample_plain(model, n, y, gen):
    """번호 y 를 모든 스텝에 넣고 표집한다. y = NULL 이면 조건 없는 표집 / sample with the index y at every step; y = NULL means unconditional."""
    x = torch.randn(n, 2, generator=gen)                                 # x_T: 표준정규 잡음에서 출발. 근사: §0 표에서 t = 100 에도 신호가 0.125 남는다 / x_T from standard normal noise; an approximation, since §0 shows 0.125 of the signal left at t = 100
    yy = torch.full((n,), y)                                             # 모든 점에 같은 번호 / the same index for every point
    for t in range(T, 0, -1):                                            # t = T, T−1, …, 1
        eps_hat = model(x, torch.full((n,), t), yy)                      # 이 스텝의 잡음 예측, 번호를 같이 넣어서 / predict this step's noise, with the index
        x = ddpm_mean(x, eps_hat, betas[t], alphas[t], abar[t])          # 평균 부분 / the mean part
        if t > 1:
            x = x + torch.sqrt(betas[t]) * torch.randn(n, 2, generator=gen)   # σ_t z. 마지막 스텝에는 더하지 않는다 / σ_t z; none on the last step
    return x

assert TARGET in range(K), "Choose one of the trained classes: 0, 1, 2."

def share_target(pts):
    return (nearest_center(pts) == TARGET).float().mean().item()

pts_u = sample_plain(model, 600, NULL, torch.Generator().manual_seed(SAMPLE_SEED))     # 조건 없음. 아래와 같은 난수(시드 101)라 두 칸의 차이는 번호 하나뿐 / no condition; the same random numbers as below (seed 101), so only the index differs
pts_c = sample_plain(model, 600, TARGET, torch.Generator().manual_seed(SAMPLE_SEED))   # 클래스 2 주문 / asked for class 2

fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.4))
for ax, pts, title in zip(axs, (pts_u, pts_c), ("no condition (y = null)", f"asked for class {TARGET} (y = {TARGET})")):
    off = nearest_center(pts) != TARGET                                  # 클래스 2 중심이 가장 가깝지 않은 점 / points not nearest to center 2
    ax.scatter(pts[~off, 0], pts[~off, 1], s=4, color=GREEN, alpha=0.55, label=f"nearest center is {TARGET}")
    ax.scatter(pts[off, 0], pts[off, 1], s=5, color=RED, alpha=0.75, label=f"nearest center is not {TARGET}")
    ax.scatter(CENTERS[:, 0], CENTERS[:, 1], marker="+", s=80, color="black", linewidths=1.5)
    ax.set(xlim=(-4, 4), ylim=(-4, 4), xticks=[-2, 0, 2], yticks=[-2, 0, 2], title=f"{title}\n{100 * share_target(pts):.0f} % near class {TARGET}"); ax.set_aspect("equal")
axs[1].legend(fontsize=7, loc="lower left", frameon=False)
plt.suptitle("600 samples each, same trained network, only the index differs", fontsize=10); plt.tight_layout(); plt.show()
print(f"조건 없음 / no condition      : 중심 {TARGET}이 가장 가까운 점 {100 * share_target(pts_u):.0f} %")
print(f"클래스 {TARGET} 주문 / asked for {TARGET}   : 중심 {TARGET}이 가장 가까운 점 {100 * share_target(pts_c):.0f} %, 퍼짐(표준편차) / spread {pts_c.std(0).mean():.2f}")
# 원본 실행 참고값 / Original-run reference (2026-11-01 v5)
#   조건 없음 / no condition   : 약 33 %   세 클래스 중 하나이니 대략 삼분의 일 / one class in three, so roughly a third
#   클래스 {TARGET} 주문 / asked for {TARGET} : 약 92 %, 퍼짐 약 0.82   겹침을 재현하는 것과 양립하는 결과; 이 비율 하나로 분포 일치를 입증하지는 않음 / compatible with reproducing overlap; this share alone does not establish distribution matching
#   100 % 가 아닙니다. 목표인 「클래스 2 의 분포」 자체가 이웃과 겹칩니다. 조건은 점을 일부만 정합니다. 더 class-specific한 표본을 원할 때 별도로 다이얼을 선택합니다.
#   Not 100 %. The target "distribution of class 2" itself overlaps its neighbors. The condition pins the point down only partly. We choose the dial separately when we want more class-specific samples.

## §4 Practice. CFG 한 줄, 그리고 w 돌리기 / The CFG line, then turn w
스텝마다 같은 $x_t$, 같은 t 에서 신경망을 **두 번** 평가합니다. 한 번은 클래스 2 를 넣고($\varepsilon_c$), 한 번은 null 을 넣고($\varepsilon_u$). 둘의 차이가 「조건이 바꾼 방향」이고, 그 방향으로 w 배 갑니다. 「The CFG Rule」 의 한 줄입니다.
*At every step the network is evaluated **twice** at the same $x_t$ and t: once with class 2 ($\varepsilon_c$), once with null ($\varepsilon_u$). Their difference is "the direction the condition changes", and we go w times along it. That is the one line of 「The CFG Rule」.*

$$\hat\varepsilon = \varepsilon_u + w\,(\varepsilon_c - \varepsilon_u)$$

빈칸은 아래 `guide` 함수의 한 줄입니다. 채우고 나면 자가검증이 두 가지를 차례로 보고, 틀리면 그 자리에서 멈춥니다. 먼저 **w = 2 · 3 의 숫자**(몸풀기 · Practice · 부호 확인)가 나오는지, 그다음 **w = 1 이 §3 의 클래스 2 표집과, w = 0 이 §3 의 조건 없는 표집과 점 하나하나 같은지**(같은 난수를 쓰므로 같아야 합니다. 소수 다섯째 자리까지). 숫자부터 보는 이유: w 대신 w × w 를 곱한 틀린 줄도 w = 0 · 1 에서는 맞는 줄과 같은 값을 냅니다(0 × 0 = 0, 1 × 1 = 1). 두 끝점만 맞춰 보면 이 실수를 놓칩니다.
*The blank is the one line of `guide` below. Once filled, the self-check looks at two things in turn and stops right there if one is wrong. First **the numbers at w = 2 and 3** (the warm-up, the Practice and the sign check), then whether **w = 1 reproduces §3's class-2 samples and w = 0 reproduces §3's unconditional samples point by point** (they use the same random numbers, so they must, to five decimals). Why the numbers come first: a wrong line that multiplies by w × w instead of w gives the same values as the right line at w = 0 and 1 (0 × 0 = 0, 1 × 1 = 1). Checking only those two end points would miss that mistake.*

두 호출은 **같은 network weights, 같은 $x_t$, 같은 t**를 씁니다. 달라지는 입력은 y와 null뿐입니다: $\varepsilon_u=\varepsilon_\theta(x_t,t,\mathrm{null})$, $\varepsilon_c=\varepsilon_\theta(x_t,t,y)$. 이 convention에서 w = 0은 unconditional, w = 1은 ordinary conditional, w > 1은 conditional 방향으로의 extrapolation입니다. **w는 sampling 때 고르고 평가하는 scale**입니다. 학습된 망이 자동으로 내놓는 보편 최적값이 아니며, 목적과 검증 결과에 따라 선택합니다.
*The two calls share the same network weights, $x_t$, and t; only y versus null differs. In this convention w = 0 is unconditional, w = 1 ordinary conditional, and w > 1 extrapolation. Choose and evaluate w at sampling time; it is not a universally optimal value automatically produced by the trained model.*

**이 절의 기호 / Notation.** $u$: unconditional; $c$: conditional; $\hat\varepsilon$: guided noise prediction; $w$: 사용자가 고르는 sampling scale. 여기서 **classifier-free**는 추가 분류기로 안내 방향을 계산하지 않는다는 뜻이며, 라벨 없이 학습한다는 뜻이 아닙니다.

**두 예측은 왜 같은 상태에서 읽나 / Why evaluate both at the same state?** $x_t$, $t$, 망의 가중치를 같게 두면 두 출력의 차이는 조건 입력을 바꾸었을 때의 예측 변화입니다. 서로 다른 표본이나 잡음 수준의 출력을 빼면 이 비교가 되지 않습니다. 이 실습은 같은 망을 두 번 호출하지만, 구현에 따라 두 입력을 한 배치로 묶어 한 번의 forward 호출에서 **두 평가**를 얻을 수도 있습니다. CFG의 본질은 비교할 두 예측과 조합 규칙이지, 함수 호출을 반드시 두 번 쓰는 방식이 아닙니다.
*Hold the noisy state, noise level, and network weights fixed so that the prediction change corresponds to the changed condition input. This lab uses two forward calls. An implementation can also batch the two inputs in one forward call; CFG requires the two evaluations and their combination, not necessarily two separate function calls.*


**v23 14장·32장 계산 연결:** $\varepsilon_u=0.1$, $\varepsilon_c=0.4$, $w=3$이면 차이 $0.3$을 세 배한 $0.9$를 $0.1$에 더해 $\hat\varepsilon=1.0$입니다. 같은 $x_t=1$, 신호·잡음 계수 $0.8,0.6$에서 $\hat x_0=(1-0.6\times1)/0.8=0.5$입니다. $w=1$이면 $\hat\varepsilon=0.4$와 $\hat x_0=0.95$가 됩니다. 이 계산은 CFG가 잡음 예측을 바꾼 뒤 표집기에 영향을 준다는 연결을 확인합니다.


In [ ]:
# 다음 이름을 씁니다 / Names used below
#   eps_u   : 조건 없는 잡음 예측 ε_u, 크기 (점 수, 2) / the unconditional noise prediction ε_u, shape (number of points, 2)
#   eps_c   : 조건 있는 잡음 예측 ε_c, 같은 크기 / the conditional noise prediction ε_c, same shape
#   w       : 안내 세기(숫자 하나) / the guidance scale (one number)
#   eps_hat : 안내한 예측 ε̂   <- 여기가 빈칸
#             the guided prediction ε̂   <- this is the blank
#
# 참고 / Reference   텐서끼리 더하고 빼고 숫자를 곱하는 것은 보통 산술 그대로입니다 / adding, subtracting tensors and multiplying by a number is plain arithmetic
#   예 / e.g.   a = torch.tensor([1., 2.]); b = torch.tensor([3., 3.]);  a + 2 * (b - a)  ->  tensor([5., 4.])

def guide(eps_u, eps_c, w):
    """「The CFG Rule」: 조건 없는 예측에서 출발해, 조건이 바꾼 방향으로 w 배 / start from the unconditional prediction, go w times along what the condition changes."""
    # TODO. 조건 없는 예측 eps_u 에서 출발해, 차이 (eps_c − eps_u) 의 w 배를 더하세요. 한 줄입니다.
    # TODO. Start from eps_u and add w times the difference (eps_c − eps_u). One line.
    eps_hat = None

    if eps_hat is None:
        raise NotImplementedError(BLANK_MSG)

    return eps_hat

@torch.no_grad()
def sample_cfg(model, n, y, w, gen):
    """스텝마다 두 번 예측하고 guide 로 합친 뒤, 9주차의 역방향 한 스텝 / predict twice per step, combine with guide, then Week 9's reverse step."""
    x = torch.randn(n, 2, generator=gen)
    yy, yn = torch.full((n,), y), torch.full((n,), NULL)
    for t in range(T, 0, -1):
        tt = torch.full((n,), t)
        eps_c = model(x, tt, yy)                                         # 조건을 넣은 예측 / with the condition
        eps_u = model(x, tt, yn)                                         # null 을 넣은 예측, 같은 x 와 t / with null, same x and t
        eps_hat = guide(eps_u, eps_c, w)                                 # 내 한 줄 / your one line
        x = ddpm_mean(x, eps_hat, betas[t], alphas[t], abar[t])
        if t > 1:
            x = x + torch.sqrt(betas[t]) * torch.randn(n, 2, generator=gen)
    return x

# 자가검증 / Check your work
a = guide(torch.tensor([0.2]), torch.tensor([0.5]), 2.0)                 # 몸풀기, w = 2 / the warm-up, w = 2
b = guide(torch.tensor([0.1]), torch.tensor([0.4]), 3.0)                 # Practice 1번, w = 3 / Practice task 1, w = 3
c = guide(torch.tensor([0.2, 0.5]), torch.tensor([0.5, 0.2]), 2.0)       # 부호 확인: 두 좌표가 반대로 움직인다 / the sign check: the two coordinates move in opposite directions
print(a, b, c)
assert torch.allclose(a, torch.tensor([0.8])) and torch.allclose(b, torch.tensor([1.0])) and torch.allclose(c, torch.tensor([0.8, -0.1])), (
    "w = 2 · 3 의 값이 기대값과 다릅니다. 아래 「기대값」 줄에서 첫 값의 증상을 찾으세요. / "
    "The values at w = 2 and 3 are not the expected ones. Find the first value's symptom in the Expected lines below.")
same_c = (sample_cfg(model, 600, TARGET, 1.0, torch.Generator().manual_seed(SAMPLE_SEED)) - pts_c).abs().max().item()
same_u = (sample_cfg(model, 600, TARGET, 0.0, torch.Generator().manual_seed(SAMPLE_SEED)) - pts_u).abs().max().item()
print(f"w = 1 과 §3 의 클래스 2 표집의 최대 차이 / largest gap between w = 1 and §3's class-2 samples : {same_c:.5f}")
print(f"w = 0 과 §3 의 조건 없는 표집의 최대 차이 / largest gap between w = 0 and §3's unconditional samples : {same_u:.5f}")
assert same_c < 1e-5 and same_u < 1e-5, (
    "w = 1 은 §3 의 클래스 2 표집과, w = 0 은 §3 의 조건 없는 표집과 같아야 합니다. §3 셀을 실행했는지 보세요. / "
    "w = 1 must reproduce §3's class-2 samples and w = 0 its unconditional samples. Check that the §3 cell was run.")
print("자가검증 통과 / self-check passed")
# 기대값 / Expected
#   tensor([0.8000]) tensor([1.]) tensor([ 0.8000, -0.1000])      (1.0 은 tensor([1.]) 로 찍힙니다 / 1.0 prints as tensor([1.]))
#     0.2 + 2 × 0.3 = 0.8 (몸풀기 / the warm-up),  0.1 + 3 × 0.3 = 1.0 (「Practice: Guide One Coordinate」 1번 / task 1)
#     [0.2 + 2 × 0.3, 0.5 − 2 × 0.3]: 첫 좌표는 커지고 둘째는 작아집니다 / the first coordinate grows, the second shrinks
#   두 최대 차이 모두 0.00000, 그리고 「자가검증 통과 / self-check passed」. w = 1 은 「조건 그대로」, w = 0 은 「조건 무시」 입니다.
#   Both gaps 0.00000, then "self-check passed". w = 1 is "the plain conditional", w = 0 is "condition ignored".
#   첫 값이 0.8000 이 아니면 / If the first value is not 0.8000:
#      0.5000  차이에 w 를 안 곱했습니다 / w was not multiplied onto the difference
#      1.1000  출발점이 eps_u 가 아니라 eps_c 입니다 / you started from eps_c instead of eps_u
#     -0.4000  차이의 부호가 반대입니다(eps_u − eps_c) / the difference has the wrong sign (eps_u − eps_c)
#      1.4000  w 를 두 번 곱했습니다(w × w). w = 0 · 1 에서는 맞는 줄과 같아서 두 차이만으로는 잡히지 않습니다
#              w was applied twice (w × w); at w = 0 and 1 it equals the right line, so the two gaps alone cannot catch it

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요. 이미 채웠다면 실행하지 마세요(내 함수를 덮어씁니다) / Only if stuck. If you already filled in the blank, skip it: it overwrites your function { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

# 다음 이름을 씁니다 / Names used below
#   eps_u   : 조건 없는 잡음 예측 ε_u, 크기 (점 수, 2) / the unconditional noise prediction ε_u, shape (number of points, 2)
#   eps_c   : 조건 있는 잡음 예측 ε_c, 같은 크기 / the conditional noise prediction ε_c, same shape
#   w       : 안내 세기(숫자 하나) / the guidance scale (one number)
#   eps_hat : 안내한 예측 ε̂   <- 여기가 빈칸
#             the guided prediction ε̂   <- this is the blank
#
# 참고 / Reference   텐서끼리 더하고 빼고 숫자를 곱하는 것은 보통 산술 그대로입니다 / adding, subtracting tensors and multiplying by a number is plain arithmetic
#   예 / e.g.   a = torch.tensor([1., 2.]); b = torch.tensor([3., 3.]);  a + 2 * (b - a)  ->  tensor([5., 4.])

def guide(eps_u, eps_c, w):
    """「The CFG Rule」: 조건 없는 예측에서 출발해, 조건이 바꾼 방향으로 w 배 / start from the unconditional prediction, go w times along what the condition changes."""
    eps_hat = eps_u + w * (eps_c - eps_u)                                # 빈칸이었던 줄. 출발점 ε_u, 방향 ε_c − ε_u, 세기 w
                                                                         # the blank: start at ε_u, direction ε_c − ε_u, strength w

    if eps_hat is None:
        raise NotImplementedError(BLANK_MSG)

    return eps_hat

@torch.no_grad()
def sample_cfg(model, n, y, w, gen):
    """스텝마다 두 번 예측하고 guide 로 합친 뒤, 9주차의 역방향 한 스텝 / predict twice per step, combine with guide, then Week 9's reverse step."""
    x = torch.randn(n, 2, generator=gen)
    yy, yn = torch.full((n,), y), torch.full((n,), NULL)
    for t in range(T, 0, -1):
        tt = torch.full((n,), t)
        eps_c = model(x, tt, yy)                                         # 조건을 넣은 예측 / with the condition
        eps_u = model(x, tt, yn)                                         # null 을 넣은 예측, 같은 x 와 t / with null, same x and t
        eps_hat = guide(eps_u, eps_c, w)                                 # 내 한 줄 / your one line
        x = ddpm_mean(x, eps_hat, betas[t], alphas[t], abar[t])
        if t > 1:
            x = x + torch.sqrt(betas[t]) * torch.randn(n, 2, generator=gen)
    return x

# 자가검증 / Check your work
a = guide(torch.tensor([0.2]), torch.tensor([0.5]), 2.0)                 # 몸풀기, w = 2 / the warm-up, w = 2
b = guide(torch.tensor([0.1]), torch.tensor([0.4]), 3.0)                 # Practice 1번, w = 3 / Practice task 1, w = 3
c = guide(torch.tensor([0.2, 0.5]), torch.tensor([0.5, 0.2]), 2.0)       # 부호 확인: 두 좌표가 반대로 움직인다 / the sign check: the two coordinates move in opposite directions
print(a, b, c)
assert torch.allclose(a, torch.tensor([0.8])) and torch.allclose(b, torch.tensor([1.0])) and torch.allclose(c, torch.tensor([0.8, -0.1])), (
    "w = 2 · 3 의 값이 기대값과 다릅니다. 아래 「기대값」 줄에서 첫 값의 증상을 찾으세요. / "
    "The values at w = 2 and 3 are not the expected ones. Find the first value's symptom in the Expected lines below.")
same_c = (sample_cfg(model, 600, TARGET, 1.0, torch.Generator().manual_seed(SAMPLE_SEED)) - pts_c).abs().max().item()
same_u = (sample_cfg(model, 600, TARGET, 0.0, torch.Generator().manual_seed(SAMPLE_SEED)) - pts_u).abs().max().item()
print(f"w = 1 과 §3 의 클래스 2 표집의 최대 차이 / largest gap between w = 1 and §3's class-2 samples : {same_c:.5f}")
print(f"w = 0 과 §3 의 조건 없는 표집의 최대 차이 / largest gap between w = 0 and §3's unconditional samples : {same_u:.5f}")
assert same_c < 1e-5 and same_u < 1e-5, (
    "w = 1 은 §3 의 클래스 2 표집과, w = 0 은 §3 의 조건 없는 표집과 같아야 합니다. §3 셀을 실행했는지 보세요. / "
    "w = 1 must reproduce §3's class-2 samples and w = 0 its unconditional samples. Check that the §3 cell was run.")
print("자가검증 통과 / self-check passed")
# 기대값 / Expected   위 TODO 셀과 같음 / the same as the TODO cell above

## §5 더 강한 guidance는 표본을 좁히고 이동시킬 수 있다 / Stronger Guidance Can Narrow and Shift the Samples
w = 0 · 1 · 3 · 6 · 12에서 600점씩 표집합니다. 모든 w에서 같은 난수(시드 101)를 써서 이 비교에서 바꾸는 것은 w뿐입니다. 다음 세 값은 서로 다른 측정입니다.
*Draw 600 samples at w = 0, 1, 3, 6, 12 with the same random numbers (seed 101). These are three different measurements.*

| 측정 / Measure | 실제 계산 / Calculation | 해석 / Interpretation |
|---|---|---|
| Share nearest to class 2 | `mean(nearest_center(pts) == TARGET)` | 중심 2가 가장 가까운 점의 비율. 분류 accuracy나 포괄적 quality가 아님 / Geometric share, not classifier accuracy or overall quality. |
| Spread | `pts.std(0).mean()` | 두 좌표의 표본 표준편차(n−1 분모)를 평균. 다양성의 제한된 요약 / Mean of the two coordinate sample SDs, a limited summary of spread. |
| Drift | `norm(pts.mean(0) - CENTERS[TARGET])` | 표본 평균과 진짜 목표 중심 사이의 Euclidean 거리 / Euclidean distance from the sample mean to the true target center. |

원본 실행에서 실제 클래스 2의 target-center share는 약 91%였습니다. 겹침까지 재현하는 조건부 표집과 더 class-specific한 표본을 요구하는 guidance는 목적이 다릅니다. **A higher target-center share does not guarantee a better match to the real distribution.** 비율이 높아지는 동안 표본이 더 좁아지거나 중심이 이동할 수 있습니다. 덱 「Stronger Guidance Can Narrow and Shift the Samples」는 원본 실행 그림과 수치를 보존합니다. 아래 출력은 현재 실행 결과입니다.
*The original real class-2 share was about 91%. Matching the conditional distribution including overlap and requesting more class-specific samples are different goals. A higher target-center share does not guarantee a better match to the real distribution. Samples can narrow or shift as that share rises. The deck preserves the original experiment; the outputs below show the current run.*

**예측 3.** w가 커지면 세 측정값이 각각 어떻게 달라질까요? 특히 drift가 계속 증가할 거라고 단정할 수 있을까요?
***Prediction 3.** How will each measure change with w? Can we assume drift must increase monotonically?*

**비교 조건 / Comparison conditions.** 망·대상 클래스·표본 수·초기 잡음과 각 스텝 잡음은 같고 $w$만 바꿉니다. 한 seed 비교는 메커니즘을 보여 주는 실험입니다. 일반적 성능 주장을 하려면 여러 seed와 해당 목적의 보류 자료가 필요합니다. `share`는 비율이고 `spread`와 `drift`는 좌표 단위의 양입니다. 평균 좌표 표준편차와 중심 거리는 각각 표본의 폭과 평균 위치의 차이를 요약합니다.


### 그림을 읽기 전에: 폭과 위치는 별개 / Width and location differ · v23 18–19장
왜 비율 하나만 보면 안 될까요? 요청한 중심 근처에 점이 모여도 **실제 분포보다 지나치게 좁거나 중심이 밀릴 수 있기 때문**입니다. 따라서 표본의 폭인 spread와 중심의 위치 차이인 drift를 따로 봅니다.

$$\mathrm{spread}=\frac{\operatorname{SD}(x_1)+\operatorname{SD}(x_2)}{2},\qquad
\mathrm{drift}=\sqrt{(\bar x_1-c_1)^2+(\bar x_2-c_2)^2}.$$

**기호:** $x_1,x_2$: 점의 첫째·둘째 좌표; $\operatorname{SD}$: 각 좌표의 표본 표준편차(분모 $n-1$); $\bar x$: 생성 표본 평균; $c$: 요청한 클래스의 진짜 중심; $n$: 표본 수. 평균은 구름의 중심, 표준편차는 그 중심 주위의 퍼짐을 나타냅니다.

**별도 숫자 예제:** 두 좌표의 표준편차가 1과 2이면 spread는 $(1+2)/2=1.5$입니다. 표본 평균이 $(0.3,0.4)$이고 진짜 중심이 $(0,0)$이면 drift는 $\sqrt{0.3^2+0.4^2}=\sqrt{0.25}=0.5$입니다. 이는 계산법을 보여 주는 가상 숫자이며 뒤의 600개 생성 표본에서 얻은 실험 수치와 다릅니다.
*Spread averages the coordinate sample SDs. Drift measures the distance from the sample mean to the true class center. The toy values below are not the experiment's measurements.*

In [ ]:
# Supplied toy arithmetic, not a new generation experiment.
toy_spread = (1.0 + 2.0) / 2
toy_mean_xy, toy_center_xy = np.array([0.3, 0.4]), np.array([0.0, 0.0])
toy_drift = np.linalg.norm(toy_mean_xy - toy_center_xy)
print(f"Spread: (1 + 2) / 2 = {toy_spread:.1f}")
print(f"Drift: sqrt(0.3^2 + 0.4^2) = {toy_drift:.1f}")
assert math.isclose(toy_spread, 1.5) and math.isclose(toy_drift, .5)


In [ ]:
# 예측 3 / Prediction 3. 빈칸 ___ 을 내 예상으로 채우세요. 맞힐 필요는 없고, 실행한 뒤 비교합니다. 이 셀은 실행하지 않아도 됩니다.
# Fill the ___ blanks with your guess. You do not need to be right; compare after running. This cell does not need to be run.
my_prediction_3 = "w 를 1 에서 6 으로 올리면 중심 2 비율은 ___, 퍼짐은 ___, 구름의 중심은 진짜 중심에서 ___ 것이다."
my_prediction_3_en = "Raising w from 1 to 6, the share near center 2 will ___, the spread will ___, and the cloud's center will ___ the true center."   # 영어로 적고 싶으면 이 줄에 / or write it in English on this line

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   WS      : 돌려 볼 w 값들 / the w values to try
#   pts     : 그 w 에서 뽑은 클래스 2 표본 600점 / the 600 class-2 samples at that w
#   share   : 클래스 2 비율. 가장 가까운 중심이 클래스 2 인 비율 / class-2 share: the share nearest to center 2
#   spread  : 두 좌표의 표본 표준편차(n−1)의 평균 / mean of two coordinate sample SDs (n−1 denominator)
#   drift   : 표본 평균과 진짜 중심 2 의 거리 / the distance between the sample mean and the true center 2

WS = (0, 1, 3, 6, 12)
real2 = data[labels == TARGET]
fig, axs = plt.subplots(1, len(WS), figsize=(2.75 * len(WS), 3.3))
print(f" w    중심 {TARGET} 최근접 비율 / share nearest to class {TARGET}   퍼짐 / spread   밀림 / drift")
print(f"data       {100 * share_target(real2):3.0f} %         {real2.std(0).mean():.2f}          {(real2.mean(0) - CENTERS[TARGET]).norm():.2f}   <- 진짜 클래스 {TARGET} / the real class {TARGET}")
rows = {}
samples_by_w = {}
for ax, w in zip(axs, WS):
    pts = sample_cfg(model, 600, TARGET, float(w), torch.Generator().manual_seed(SAMPLE_SEED))   # 다섯 w 모두 같은 난수. w = 0 · 1 은 §3 의 두 표집 그대로 / the same random numbers for every w; w = 0 and 1 are §3's two samples
    share, spread, drift = share_target(pts), pts.std(0).mean().item(), (pts.mean(0) - CENTERS[TARGET]).norm().item()
    rows[w] = (share, spread, drift)
    samples_by_w[w] = pts.clone()                                                          # 아래 확대 그림이 같은 점을 다시 쓴다 / the zoom below reuses these exact points
    print(f"{w:3d}        {100 * share:3.0f} %         {spread:.2f}          {drift:.2f}")
    ax.scatter(real2[:, 0], real2[:, 1], s=2, color=GRAY, alpha=0.35)                      # 진짜 클래스 2 를 회색으로 깔고 / the real class 2 in gray underneath
    ax.scatter(pts[:, 0], pts[:, 1], s=4, color=GREEN, alpha=0.55)
    ax.scatter(CENTERS[:, 0], CENTERS[:, 1], marker="+", s=70, color="black", linewidths=1.4)
    ax.set(xlim=(-4, 4), ylim=(-4, 4), xticks=[-2, 0, 2], yticks=[-2, 0, 2]); ax.set_aspect("equal")
    ax.set_title(f"w = {w}\n{100 * share:.0f} % near {TARGET} · spread {spread:.2f} · drift {drift:.2f}", fontsize=9)
plt.suptitle(f"600 samples asking for class {TARGET} at each w, same random numbers (gray = real class {TARGET}, + = centers)", fontsize=10); plt.tight_layout(); plt.show()
# 원본 실행 참고값 / Original-run reference (2026-11-01 v5; 덱의 그림·수치와 동일 / the deck's original experiment)
#    w    중심 2 최근접 비율 / share nearest to class 2   퍼짐 / spread   밀림 / drift
#   data   91 %         0.81          0.02
#    0     33 %         1.28          1.41
#    1     92 %         0.82          0.10
#    3     100 %         0.48          0.90
#    6     100 %         0.32          1.03
#   12     100 %        0.20         1.01
#   클래스 2 비율은 w = 3 에서 이미 100 % 로 진짜 클래스 2(91 %)를 넘습니다. 「더 정확」 이 아니라 겹치는 자리를 떠난 것입니다.
#   The class-2 share is already 100 % at w = 3, above the real class 2 (91 %): not "more correct", the samples have left the overlap.
#   퍼짐은 계속 줄어듭니다: 1.28 → 0.82 → 0.48 → 0.32 → 0.20(진짜 0.81). 구름의 중심은 w = 1 에서 진짜 중심에 가장 가깝고(밀림 0.10), w = 3 부터 이웃에서 먼 쪽으로 밀려나 진짜 중심을 지나칩니다(0.90 · 1.03 · 1.01, w 와 함께 늘기만 하지는 않습니다).
#   The spread goes 1.28 → 0.82 → 0.48 → 0.32 → 0.20 (real 0.81). The cloud's center is closest to the true center at w = 1 (drift 0.10); from w = 3 on it is pushed away from the neighbors, past the true center (0.90, 1.03, 1.01; it does not simply grow with w).

### 같은 두 구름을 확대해서 / The same two clouds, zoomed in
왼쪽 w = 1, 오른쪽 w = 6 이고 위 셀의 점을 그대로 씁니다(새로 학습하거나 뽑지 않음). **회색은 진짜 클래스 2, 초록은 만든 점, 검은 + 는 진짜 중심, 금색 별은 만든 점의 평균**입니다. 초록 구름의 넓이가 퍼짐(spread), + 에서 별까지의 화살표가 밀림(drift)입니다. 두 칸의 축 범위는 같습니다.
*Left w = 1, right w = 6, with the exact points of the cell above (nothing new is trained or drawn). **Gray = the real class 2, green = the generated points, black + = the true center, gold star = the generated mean.** The green cloud's width is the spread; the arrow from + to the star is the drift. Both panels share the same axes.*

먼저 「오른쪽 구름이 더 좁은가?」, 그다음 「별이 + 와 같은 자리인가?」 를 따로 답하세요. 덱 「Does a Tighter Cloud Mean a Better Cloud?」 의 그림이 이 셀입니다. 이미지가 과하게 안내될 때의 문제를 떠올리는 비유이지, 이 2차원 그림이 이미지의 과포화를 잰 것은 아닙니다.
*Answer two questions separately: "Is the right cloud narrower?", then "Is the star where the + is?". The figure on "Does a Tighter Cloud Mean a Better Cloud?" is this cell. It is an analogy for over-guided images, not a measurement of image oversaturation.*

**그림의 기호 / Plot key.** $w$: guidance scale; 회색: 실제 클래스 2; 초록: 생성 표본; +: 진짜 목표 중심; 별: 생성 표본 평균; 화살표 길이: drift. 동일한 축 범위에서 넓이와 중심 위치를 따로 읽습니다.


In [ ]:
# 다음 이름을 씁니다 / Names used below
#   samples_by_w : 위 셀이 w 마다 저장한 600점 / the 600 points the cell above kept for each w
#   mean         : 만든 점의 평균(금색 별) / the generated mean (gold star)

fig, axs = plt.subplots(1, 2, figsize=(7.0, 3.6), dpi=130)
for ax, w in zip(axs, (1, 6)):
    pts = samples_by_w[w]
    center, mean = CENTERS[TARGET], samples_by_w[w].mean(0)
    ax.scatter(real2[:, 0], real2[:, 1], s=6, color=GRAY, alpha=0.3, label=f"real class {TARGET}")
    ax.scatter(pts[:, 0], pts[:, 1], s=8, color=GREEN, alpha=0.55, label="generated")
    ax.scatter(*center.tolist(), marker="+", s=140, color="black", linewidths=2, label="true center")
    ax.scatter(*mean.tolist(), marker="*", s=150, color=GOLD, edgecolors="black", linewidths=0.5, zorder=5, label="generated mean")
    ax.annotate("", xy=mean.tolist(), xytext=center.tolist(), arrowprops=dict(arrowstyle="->", color=RED, lw=2))   # 밀림 / the drift
    ax.set(xlim=(-2, 4), ylim=(-4, 2)); ax.set_aspect("equal")
    ax.set_xlabel("coordinate 1", fontsize=11); ax.set_ylabel("coordinate 2", fontsize=11); ax.tick_params(labelsize=10)
    ax.set_title(f"w = {w}: spread {rows[w][1]:.2f}, drift {rows[w][2]:.2f}", fontsize=12)
fig.legend(*axs[0].get_legend_handles_labels(), loc="lower center", ncol=4, fontsize=9, frameon=False)   # 범례는 그림 아래, 점을 가리지 않게 / legend below, off the dots
fig.suptitle("Cloud width = spread; true center to generated mean = drift", fontsize=12)          # 그림 글은 영어만 / plot text in English only
plt.tight_layout(rect=(0, 0.07, 1, 1)); plt.show()
print(f"w = 1: 퍼짐 / spread {rows[1][1]:.2f}, 밀림 / drift {rows[1][2]:.2f}     w = 6: 퍼짐 / spread {rows[6][1]:.2f}, 밀림 / drift {rows[6][2]:.2f}     진짜 클래스 {TARGET} 의 퍼짐 / real class-{TARGET} spread {real2.std(0).mean():.2f}")
# 원본 실행 참고값 / Original-run reference (2026-11-01 v5)
#   w = 1: 퍼짐 / spread 0.82, 밀림 / drift 0.10     w = 6: 퍼짐 / spread 0.32, 밀림 / drift 1.03     진짜 / real 0.81
#   w = 6 은 더 좁지만 진짜 중심에서 더 멀어졌습니다. 이 예에서는 w = 1 이 진짜 클래스 2 의 퍼짐과 중심에 더 가깝습니다.
#   w = 6 is narrower but further from the true center. Here w = 1 is closer to the real class 2 in both spread and center.

### 같은 표본으로 \(w\)를 골라 보기 / Choose a scale for a stated goal · v23 18–20장

**공정하게 비교하기.** 위 sweep은 \(w\)마다 같은 표집 시드로 난수 생성기를 다시 시작합니다. 시작 잡음과 각 역방향 단계의 새 잡음이 같고, \(w\)만 달라집니다. 아래 표는 위에서 저장한 점의 지표를 다시 읽습니다.

| 목표 | 판단 기준 |
|---|---|
| 요청 중심 영역에 가능한 한 많이 보내기 | **Share 최대**. 여러 중심 중 목표 클래스 중심이 가장 가까운 점의 비율입니다. |
| 원래 클래스 분포의 폭과 위치 재현하기 | **Width gap** \(=|\text{생성 spread}-\text{기준 spread}|\)과 **Drift** \(=\|\text{생성 평균}-\text{참 중심}\|\)을 각각 작게 합니다. |

기준 클래스의 점들도 이웃 중심에 더 가까울 수 있습니다. 그래서 share 100%를 목표로 하면 겹치는 영역의 점을 덜 만들 수 있습니다. 폭과 위치가 서로 다른 \(w\)를 가리키면 어느 쪽이 중요한지에 따라 선택이 달라집니다. 분포 전체를 평가하려면 구름의 모양과 겹침도 확인합니다.

*Reuse the same noise draws at every scale. Maximize nearest-center share for the first goal. For the second goal, compare the reference width and center using two separate measures.*


In [ ]:
# Reuse the sweep above. Each row comes from 600 samples with the same seed.
reference_spread = real2.std(0).mean().item()
reference_share = share_target(real2)
width_gap = {w: abs(spread - reference_spread) for w, (_, spread, _) in rows.items()}
drift_by_w = {w: drift for w, (_, _, drift) in rows.items()}
share_by_w = {w: share for w, (share, _, _) in rows.items()}

best_share = max(share_by_w.values())
share_winners = [w for w in rows if share_by_w[w] == best_share]
best_width_gap = min(width_gap.values())
best_drift = min(drift_by_w.values())
width_winners = [w for w in rows if math.isclose(width_gap[w], best_width_gap)]
drift_winners = [w for w in rows if math.isclose(drift_by_w[w], best_drift)]
joint_winners = [w for w in width_winners if w in drift_winners]

print(f"Class {TARGET} | sampling seed {SAMPLE_SEED} | 600 per scale")
print(f"Reference: share {reference_share:.1%}, spread {reference_spread:.3f}")
print(f"{'w':>4}  {'share ↑':>8}  {'spread':>8}  {'width gap ↓':>12}  {'drift ↓':>8}")
for w, (share, spread, drift) in rows.items():
    print(f"{w:4g}  {share:8.1%}  {spread:8.3f}  {width_gap[w]:12.3f}  {drift:8.3f}")
print()
print(f"Highest nearest-center share: {share_winners}")
print(f"Smallest scale among share winners: {min(share_winners)}")
print(f"Closest width: {width_winners} | Closest center: {drift_winners}")
if joint_winners:
    print(f"Best on both width and center: {joint_winners}")
else:
    print("Width and center favor different scales. Decide which matters more for the task.")


**기본 실행 읽기.** 시드 101, 클래스 2의 기준 share는 약 91%, spread는 약 0.81입니다. \(w=3,6,12\)는 share 100%로 동률입니다. 동률일 때 가장 작은 배율을 고르기로 했다면 \(w=3\)입니다. \(w=1\)의 share는 약 92%이지만, 폭 차이는 약 0.01, drift는 0.10으로 둘 다 후보 중 가장 작습니다. 이 실험에서 폭과 중심을 재현하려면 \(w=1\)을 고를 수 있습니다.

**이미지 생성에서는? · v23 20장 「What Guidance Can Break」.** 프롬프트와 시드 목록을 고정하고 여러 \(w\)로 이미지를 뽑은 뒤, 요청을 얼마나 잘 따르는지, 이미지가 자연스러운지, 이미지 특징이 얼마나 다양한지를 평가합니다. 위 표처럼 코드로 생성과 지표 계산을 반복할 수 있습니다. 개발자는 목적에 맞는 설정을 기본값으로 정할 수 있고, 도구가 옵션을 제공하면 사용자도 바꿀 수 있습니다. 예를 들어 Diffusers에서는 guidance_scale로 지정합니다. [Diffusers 문서](https://huggingface.co/docs/diffusers/api/pipelines/stable_diffusion/text2img)

*With the default settings, share is tied at 3, 6 and 12. The smallest-scale tie-break gives 3. Both width and center favor 1. For images, compare prompt fit, visual quality and feature diversity. Developers can choose defaults and users can adjust exposed settings.*


**21–22장 연결 / From points to images.** 21장의 고양이 사진은 공간 위치가 많은 예이고, 오른쪽은 작은 특징값 격자입니다. CFG는 각 단계 같은 상태에서 요청을 주거나 빼고 두 번 예측합니다. 22장은 학습 때 이미지에서 격자로 가는 인코더와 생성 마지막에 격자를 이미지로 바꾸는 디코더를 보여 줍니다. 이 실습의 계산 대상은 좌표 두 개인 점입니다.


## 정리. 예측 다시 읽기 / Wrap-up. Re-read your predictions
1. **§1 · §3.** 실제 클래스 2 자체가 이웃과 겹칩니다. 원본 실행에서 실제 자료 91%, 조건부 표본 92%가 중심 2에 가장 가까웠습니다. 겹침은 그 자체로 모형 오류가 아니며, 이 비율만으로 전체 분포 일치를 입증하지도 않습니다. 「Can We Prefer More Class-Specific Samples?」.
   *§1 · §3. The real class 2 itself overlaps its neighbors. The original run had shares of 91% for the real data and 92% for conditional samples. Overlap is not itself a model error, and this share alone does not establish full distribution matching. Slide 「Can We Prefer More Class-Specific Samples?」.*
2. **§4.** 안내는 한 줄입니다. 같은 $x_t$, 같은 t 에서 두 번 예측하고, 차이의 w 배를 조건 없는 예측에 더합니다. w = 1 이 보통의 조건부 표집, w = 0 이 조건 무시입니다(점 하나하나 같았습니다). 「The CFG Rule」.
   *§4. Guidance is one line: predict twice at the same $x_t$ and t, and add w times the difference to the unconditional prediction. w = 1 is plain conditional sampling and w = 0 ignores the condition (identical point by point). Slide 「The CFG Rule」.*
3. **§5.** 원본 실행에서 w 를 올리면 중심 2 최근접 비율은 진짜 자료(91 %)를 넘어 오르고(w = 3 에서 100 %) 퍼짐은 줄며, w = 3 부터 구름이 이웃에서 먼 쪽으로 밀려나 진짜 중심을 지나칩니다. 이 밀림은 이미지의 과포화를 떠올리는 비유이며 이미지 품질을 측정한 결과가 아닙니다. 더 높은 target-center share가 더 좋은 분포 일치를 보장하지 않습니다. w 는 공짜가 아니라 고르고 보고해야 하는 값입니다. 「Stronger Guidance Can Narrow and Shift the Samples」.
   *§5. In the original run, raising w pushes the target-center share past the real data's 91 % (100 % by w = 3), shrinks the spread, and from w = 3 on pushes the cloud away from the neighbors, past the true center. This drift is an analogy for over-guided images, not a measurement of image quality. A higher target-center share does not guarantee a better distribution match. w is not free: it is a value you choose and report. Slide 「Stronger Guidance Can Narrow and Shift the Samples」.*

**프로젝트 Track 3로 이어 가기 / Continue with Project Track 3.** 강의 29장 「Assignment 3 and the Project」와 별도 `FP_Track3_guidance_scale_starter_v7.ipynb`가 연결됩니다. 이 비채점 실습을 그대로 과제3로 제출하는 것이 아닙니다. Track 3에서는 기본 조건에 비해 **중심 반지름 하나** 또는 **학습 update 수 하나**만 바꾼 뒤 같은 w sweep을 반복합니다. starter에서 class distance라고 부르는 값은 원점에서 중심까지의 반지름입니다. 정삼각형에서 두 중심 사이는 그 값의 $\sqrt{3}$배입니다.

기본 반지름1.5·3,000update에 대해 반지름1.0/2.5 또는 학습1,000/6,000update 중 한 조건을 선택합니다. 재학습 seed29·30·31을 각 조건에 공통으로 쓰고, sampling seed101·600표본·w=0,1,3,6,12·잡음 스케줄·drop확률0.15를 고정합니다. **학습 seed 반복은 망을 처음부터 다시 학습하는 반복**이며, 같은 망에서 sampling seed만 바꾼 것과 다릅니다. 실행 전에 질문·변경 조건·고정 조건·판정 기준을 써 두고, 결과 뒤에는 실제 관찰과 한계를 구분합니다. 아래 분류기 utility는 별도 응용 제안이며 Track 3에 반드시 추가해야 하는 과제가 아닙니다.
*Use the separate Track 3 starter. Change one condition, repeat the same guidance sweep under matched training seeds, and keep the other settings fixed. Training-seed repeats retrain the model. The classifier application below is a separate proposed experiment, not a required extra task for Track 3.*

**Proposed experiment, not measured in today’s lab / 오늘 측정하지 않은 제안 실험.** 이 실습은 점을 **만드는** 일까지입니다. 만든 점이 분류기에 도움이 되는지(희소 클래스의 held-out recall)는 다른 실험이고, 덱 「Testing Whether Guidance Helps a Classifier」 와 「Proposed Classifier Experiment」 가 그 설계입니다. 학습용 실제 자료로 생성기를 학습하고, 실제 자료(합성을 더하거나 더하지 않고)로 분류기를 학습한 뒤, **보지 않은 실제 자료**로 분류기를 평가합니다. w 는 검증용 실제 자료에서 고르고, 시험용 실제 자료는 마지막에 한 번만 씁니다.
***Proposed experiment, not measured in today’s lab.** This lab stops at **making** points. Whether the points help a classifier (held-out recall on the rare class) is a separate experiment, designed on "Testing Whether Guidance Helps a Classifier" and "Proposed Classifier Experiment". Train the generator on the real training split, train a classifier on real data with or without synthetic additions, then evaluate the classifier on **real data it has never seen**. Choose w on a real validation split, and use the real test split once, at the end.*

**설명용 계산 / An illustrative calculation.** 실제 희소 사례 20 개 중 16 개를 잡으면 recall = 16 / 20 = 80 % 입니다. w 들을 비교할 때는 오경보율(정상 사례를 잘못 경보하는 비율)을 같게 맞춥니다. **이 숫자는 설명용이며 이 실습의 결과가 아닙니다.**
*Catching 16 of 20 real rare cases is a recall of 16 / 20 = 80 %. When comparing values of w, hold the false-alarm rate (normal cases wrongly flagged) at the same level. **These numbers are illustrative, not a result of this lab.***

**왜 여기서 LDM이 등장하나 / Why introduce latent diffusion now?** 지금은 좌표 두 개를 갱신하므로 계산이 작습니다. 이미지에서는 많은 위치를 여러 표집 스텝에서 반복 처리하고 CFG를 위해 두 예측도 평가합니다. 이 비용 때문에 LDM은 학습한 autoencoder로 더 작은 표현을 만들고 그 공간에서 확산을 수행한 뒤 이미지를 복원합니다. 이것은 $w$를 더 크게 하는 규칙이 아니라 **계산 공간을 바꾸는 선택**입니다. 압축으로 줄어드는 계산과 잃을 수 있는 세부정보를 함께 고려합니다. 이 노트북에서는 이미지 압축이나 LDM을 실행하지 않습니다.
*Updating two coordinates is cheap. Images require repeated computation over many positions and two CFG evaluations. Latent diffusion addresses that cost by denoising a smaller learned representation and decoding it. It changes the computation space, not the guidance scale, and trades computation against information retained by compression. This notebook does not run an image autoencoder or LDM.*

**마지막 큰 그림 / Final perspective.** Conditioning은 **무엇을 요청하는가**, CFG는 **그 요청의 차이를 얼마나 강조하는가**, DDPM은 **예측을 다음 상태로 어떻게 바꾸는가**, LDM은 **어느 표현 공간에서 계산하는가**를 다룹니다. 서로를 대체하는 우열 순서가 아닙니다. 이 실습은 2차원 조건부 DDPM과 CFG를 실행했고, 텍스트 인코더·cross-attention·이미지 autoencoder·LDM 및 분류기 utility 실험은 실행하지 않았습니다. 프로젝트에서는 실제 수행한 실험과 제안한 실험을 분리해서 씁니다.
*These are different design choices, not a ranking. This notebook runs conditional DDPM and CFG on 2-D points. It does not run a text encoder, cross-attention, an image autoencoder, latent diffusion, or a downstream classifier utility experiment. Report performed and proposed experiments separately.*

### v23 슬라이드와 실습 매칭 / Slide-to-lab map
| 실습의 위치 / Lab location | v23 슬라이드 번호와 제목 / Slides |
|---|---|
| 기호·복습 / Symbols and recall | 4. Which Noise Is Known, and Which Is Predicted?<br>5. Week 9 in Three Lines<br>6. Conditioning Gives the Network a Request<br>7. What Does “No Label” Mean to the Network? |
| 몸풀기·CFG 계산 / Warm-up and CFG | 10. Two Predictions, One Shared Network<br>11. CFG on One Number<br>12. The CFG Rule<br>13. What the Sampler Does With the Guided Noise<br>14. Practice: Guide One Coordinate |
| §1·§3 분포와 요청 / Data and condition | 8. What Do the Dots in Today’s Pictures Show?<br>9. Can We Prefer More Class-Specific Samples? |
| §2 학습 준비 / Prepare the two modes | 15. How Information Flows<br>16. Training Steps or Sampling Steps?<br>17. Training Changes by One Line |
| §4 한 줄 / The one-line practice | 12. The CFG Rule<br>14. Practice: Guide One Coordinate |
| §5 세 지표 / Three measurements | 18. Does a Tighter Cloud Mean a Better Cloud?<br>19. Stronger Guidance Can Narrow and Shift the Samples<br>20. What Guidance Can Break |
| 실습 범위 밖의 이미지 경로 / Image path, not implemented | 21. From One Point to a Whole Image<br>22. Run Diffusion on a Smaller Learned Representation<br>23. How Does Text Reach the Denoiser? |
| 실제 용도 검증 제안 / Proposed utility test | 24. Testing Whether Guidance Helps a Classifier<br>36. Proposed Classifier Experiment |
| Track 3·실습·마무리 / Project, lab and wrap-up | 29. Assignment 3 and the Project<br>27. Lab. What Happens in Each Section<br>28. Different Choices Change Different Parts |


| 추가 계산 / Added arithmetic | v23 슬라이드 번호와 제목 / Slides |
|---|---|
| Forward 혼합과 손실 / Forward mixture and loss | 4. Which Noise Is Known, and Which Is Predicted?<br>5. Week 9 in Three Lines |
| 잡음 기여 → 남은 신호 → 원본 추정 / Noise contribution → remaining signal → clean estimate | 13. What the Sampler Does With the Guided Noise |
| 별도 숫자의 정답 / Worked practice answer | 14. Practice: Guide One Coordinate<br>32. Practice Answers: Guide One Coordinate |
| 폭과 중심 계산 / Spread and drift arithmetic | 18. Does a Tighter Cloud Mean a Better Cloud?<br>19. Stronger Guidance Can Narrow and Shift the Samples |
| 선택 참고 / Optional score connection | 34. Optional A · Classifier Guidance Came First<br>35. Optional B · CFG as a Score |


**선택 기준 실습 / Scale selection (§5):** v23 20장 「What Guidance Can Break」에서 2D 실험과 실제 이미지 생성의 평가 절차를 연결합니다.


## 참고. 원본 추정값과 다음 상태는 다르다 / Reference: a clean estimate is not the next state
본 실습은 `ddpm_mean`에 잡음 예측을 바로 넣습니다. 앞에서 본 원본 추정값은 어디에 쓰인 걸까요? 아래는 **실험 수치를 바꾸지 않는 별도 숫자 예제**입니다. 본문의 계산이 이해되면 이 참고는 건너뛰어도 됩니다.
*The main code feeds the noise prediction directly into `ddpm_mean`. The calculation below connects that implementation to the clean estimate. It is a separate toy example, not a new experimental setting.*

현재 값 $x_t=1$, guided noise $\hat\varepsilon=0.8$, $\bar\alpha_t=0.64$이면 신호·잡음 계수는 0.8과 0.6입니다. 먼저 $\hat x_0=(1-0.6\times0.8)/0.8=0.65$를 얻습니다. 이는 깨끗한 원본의 **추정값**입니다.

이번 설명용 한 스텝에서만 $\beta_t=0.1$, $\alpha_t=0.9$를 가정하면 다음 상태의 평균은
$$\mu_\theta=\frac{1-(0.1/0.6)\times0.8}{\sqrt{0.9}}\approx0.9135.$$
새 잡음의 가상 실현값을 $z=0.2$로 주면 이 실습의 DDPM 표집 규칙은
$$x_{t-1}=\mu_\theta+\sqrt{0.1}\times0.2\approx0.9768.$$
**0.65, 0.9135, 0.9768은 서로 다른 양**입니다. 원본 추정값, 다음 상태의 평균, 잡음까지 더한 다음 상태를 구분하세요. 실제 마지막 스텝에서는 새 잡음을 더하지 않습니다.

**Notation.** $\hat x_0$: clean-data estimate; $\mu_\theta$: next-state mean; $x_{t-1}$: next noisy state; $z$: fresh standard-normal draw, here a supplied toy value; $\beta_t$: this implementation's sampling variance for a nonfinal step. $\bar\alpha_t=\alpha_t\bar\alpha_{t-1}$.


**v23 연결:** 13장과 15장. 앞의 0.48 → 0.52 → 0.650 계산에서 멈추지 않고, 왜 다음 noisy state를 만들기 위한 별도 규칙이 필요한지 확인하는 참고입니다.


In [ ]:
# Optional arithmetic only. No random draws and no changes to model or schedule.
# The main experiment remains T=100, seed=29 for training and seed=101 for sampling.
toy_xt, toy_eps, toy_abar, toy_beta, toy_z = 1.0, 0.8, 0.64, 0.1, 0.2
toy_alpha = 1.0 - toy_beta
toy_signal = math.sqrt(toy_abar)
toy_noise = math.sqrt(1.0 - toy_abar)
toy_x0 = (toy_xt - toy_noise * toy_eps) / toy_signal
toy_mu = (toy_xt - toy_beta * toy_eps / toy_noise) / math.sqrt(toy_alpha)
toy_next = toy_mu + math.sqrt(toy_beta) * toy_z

# Substitute the clean estimate into the equivalent posterior-mean expression.
toy_prev_abar = toy_abar / toy_alpha
toy_mu_from_x0 = (
    toy_beta * math.sqrt(toy_prev_abar) / (1-toy_abar) * toy_x0
    + math.sqrt(toy_alpha) * (1-toy_prev_abar) / (1-toy_abar) * toy_xt
)
assert math.isclose(toy_x0, 0.65, abs_tol=1e-12)
assert math.isclose(toy_mu, toy_mu_from_x0, abs_tol=1e-12)
print(f"Clean estimate: {toy_x0:.4f}; next-state mean: {toy_mu:.4f}; next state: {toy_next:.4f}")
print("Same mean from the clean estimate:", math.isclose(toy_mu, toy_mu_from_x0, abs_tol=1e-12))


## 선택 참고. 잡음 예측을 score로 다시 읽기 / Optional score view · v23 34–35장
**필수 빈칸과 2차원 실험은 위에서 끝났습니다.** 여기서는 CFG가 왜 “방향을 강조한다”고 설명되는지, 강의의 참고 식과 이미 배운 잡음 예측을 연결합니다. 추가 망이나 분류기를 학습하지 않습니다.

Score는 $s_t=\nabla_{x_t}\log p_t(x_t)$, 즉 현재 잡음 수준에서 **로그 밀도가 국소적으로 증가하는 방향**입니다. $\nabla_{x_t}$는 망의 가중치가 아니라 입력 좌표에 대한 미분입니다. 잡음 예측으로부터 만든 score 추정은
$$\hat s_u=-\varepsilon_u/\sigma_t,\quad \hat s_c=-\varepsilon_c/\sigma_t,\quad
\hat s_g=\hat s_u+w(\hat s_c-\hat s_u)=-\hat\varepsilon/\sigma_t.$$

**기호:** $p_t$: 잡음 수준 $t$에서의 밀도; $s$: score; 모자 $\hat s$: 망에서 얻은 추정; $u,c,g$: 조건 없음·조건 있음·guidance; $\sigma_t=\sqrt{1-\bar\alpha_t}>0$: **forward에서 섞인 잡음의 계수**입니다. 역방향 한 스텝에서 새 잡음에 곱하는 $\sqrt{\beta_t}$와 다른 양입니다. 오늘 코드의 100스텝 중 하나를 지정한 예가 아닙니다.

숫자를 넣으면 $\sigma_t=0.6$, $\varepsilon_u=0.2$, $\varepsilon_c=0.5$에서 $\hat s_u=-0.2/0.6=-1/3$, $\hat s_c=-0.5/0.6=-5/6$입니다. $w=2$이면 $\hat s_g=-1/3+2(-5/6+1/3)=-4/3$. 앞에서 얻은 guided noise 0.8을 $-0.8/0.6$으로 바꾼 결과와 같습니다. 이것은 **같은 두 예측을 다른 단위로 쓴 산술적 동등성**입니다.

참고로 실제 밀도에 대한 Bayes 항등식은 $s_c=s_u+\nabla_{x_t}\log p_t(y\mid x_t)$입니다. 학습된 score는 실제 score의 근사이므로 두 망 예측의 차이를 실제 분류기 gradient와 무조건 같다고 단정하지 않습니다. 이 한 시점의 식만으로 최종 생성분포의 모양이 정확히 보장되는 것도 아닙니다.
*This optional arithmetic links estimated noise and estimated scores. Bayes' identity applies to true density scores; learned estimates are approximate. It does not by itself establish an exact final sampling distribution.*

In [ ]:
# Optional supplied arithmetic only; does not train or sample another model.
toy_sigma, toy_score_w = 0.6, 2.0
toy_score_u, toy_score_c = -0.2 / toy_sigma, -0.5 / toy_sigma
toy_score_g = toy_score_u + toy_score_w * (toy_score_c - toy_score_u)
toy_score_from_noise = -(0.2 + toy_score_w * (0.5 - 0.2)) / toy_sigma
print(f"Estimated scores: u={toy_score_u:.6f}, c={toy_score_c:.6f}, guided={toy_score_g:.6f}")
print(f"From guided noise: -0.8 / 0.6 = {toy_score_from_noise:.6f}")
assert math.isclose(toy_score_g, -4 / 3)
assert math.isclose(toy_score_g, toy_score_from_noise)


In [ ]:
# 실행 환경 기록 / Record the runtime
import sys, platform, datetime
print("Runtime :", platform.platform(), "| Python", sys.version.split()[0])
print("torch   :", torch.__version__, "| threads", torch.get_num_threads(), "|", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("Run at  :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))

print("numpy   :", np.__version__)
print("matplotlib:", __import__("matplotlib").__version__)
print("Model device:", next(model.parameters()).device)
print("Settings: T=", T, "; P_DROP=", P_DROP, "; data seed=", DATA_SEED, "; training seed=", TRAIN_SEED, "; sample seed=", SAMPLE_SEED, "; 3000 Adam updates")
